In [1]:
# Libraries Kaggle does not include: roboflow (dataset download), ultralytics (YOLO model)
!pip install -q roboflow ultralytics

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

api_key = UserSecretsClient().get_secret("roboflow")
rf = Roboflow(api_key=api_key)
project = rf.workspace("sample-wy2mp").project("conveyor-belt-damage")
dataset = project.version(1).download("yolov11", location="/kaggle/working/belt_damage")

print(dataset.location)
!cat {dataset.location}/data.yaml

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/belt_damage in yolov11:: 100%|██████████| 1856/1856 [00:00<00:00, 9781.01it/s]


/kaggle/working/belt_damage
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 8
names: ['Hole', 'Human', 'Other Objects', 'Puncture', 'Roller', 'Tear', 'impact damage', 'patch work']

roboflow:
  workspace: sample-wy2mp
  project: conveyor-belt-damage
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/sample-wy2mp/conveyor-belt-damage/dataset/1

In [3]:
import glob, collections

names = ['Hole', 'Human', 'Other Objects', 'Puncture', 'Roller', 'Tear', 'impact damage', 'patch work']
for split in ["train", "valid", "test"]:
    files = glob.glob(f"/kaggle/working/belt_damage/{split}/labels/*.txt")
    counts = collections.Counter()
    outlines = boxes = 0
    for f in files:
        for line in open(f):
            parts = line.split()
            if parts:
                counts[names[int(parts[0])]] += 1
                if len(parts) > 5:
                    outlines += 1
                else:
                    boxes += 1
    print(split, "-", len(files), "photos,", outlines, "outlines,", boxes, "boxes")
    for name in names:
        print("   ", name, counts[name])

train - 645 photos, 1776 outlines, 0 boxes
    Hole 33
    Human 89
    Other Objects 99
    Puncture 83
    Roller 514
    Tear 467
    impact damage 210
    patch work 281
valid - 185 photos, 493 outlines, 0 boxes
    Hole 7
    Human 26
    Other Objects 21
    Puncture 24
    Roller 122
    Tear 169
    impact damage 42
    patch work 82
test - 92 photos, 201 outlines, 0 boxes
    Hole 2
    Human 3
    Other Objects 10
    Puncture 13
    Roller 45
    Tear 55
    impact damage 31
    patch work 42


In [4]:
# Keep only the five belt-damage classes; drop Human, Roller and Other Objects, which are not belt wear
import glob, os

ROOT = "/kaggle/working/belt_damage"
old_names = ['Hole', 'Human', 'Other Objects', 'Puncture', 'Roller', 'Tear', 'impact damage', 'patch work']
keep = ['Tear', 'patch work', 'Puncture', 'Hole', 'impact damage']

if os.path.exists(f"{ROOT}/belt.yaml"):
    print("already done, nothing changed")
else:
    kept = dropped = 0
    for path in glob.glob(f"{ROOT}/*/labels/*.txt"):
        new_lines = []
        for line in open(path):
            parts = line.split()
            if not parts:
                continue
            name = old_names[int(parts[0])]
            if name in keep:
                parts[0] = str(keep.index(name))
                new_lines.append(" ".join(parts) + "\n")
                kept += 1
            else:
                dropped += 1
        with open(path, "w") as f:
            f.writelines(new_lines)

    with open(f"{ROOT}/belt.yaml", "w") as f:
        f.write(f"path: {ROOT}\ntrain: train/images\nval: valid/images\ntest: test/images\nnames:\n")
        for number, name in enumerate(keep):
            f.write(f"  {number}: {name}\n")
    print("kept", kept, "outlines, dropped", dropped)

!cat {ROOT}/belt.yaml

kept 1541 outlines, dropped 929
path: /kaggle/working/belt_damage
train: train/images
val: valid/images
test: test/images
names:
  0: Tear
  1: patch work
  2: Puncture
  3: Hole
  4: impact damage


In [5]:
# Train the segmentation model on the five belt-damage classes
import os
os.environ["WANDB_MODE"] = "disabled"
from ultralytics import YOLO

model = YOLO("yolo26s-seg.pt")
model.train(data="/kaggle/working/belt_damage/belt.yaml", epochs=60, imgsz=640, batch=16,
            patience=15, seed=0, project="/kaggle/working/runs", name="belt_seg", exist_ok=True)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/belt_damage/belt.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


       2/60      6.73G      1.672      2.668      4.336    0.02025       4.19         17        640: 100% ━━━━━━━━━━━━ 41/41 2.5it/s 16.4s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.0it/s 3.1s0.6s
                   all        185        324       0.55      0.272      0.292      0.147      0.532       0.24      0.232     0.0753

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
       3/60      6.73G      1.851       2.67      2.635    0.02334      3.124         25        640: 100% ━━━━━━━━━━━━ 41/41 2.4it/s 17.4s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.9it/s 3.1s0.7s
                   all        185        324       0.27      0.323      0.246      0.103      0.212      0.309      0.189     0.0575



ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d71800e9c50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,  

In [6]:
from ultralytics import YOLO

best = YOLO("/kaggle/working/runs/belt_seg/weights/best.pt")
metrics = best.val(data="/kaggle/working/belt_damage/belt.yaml", split="test",
                   project="/kaggle/working/runs", name="belt_seg_test", exist_ok=True)
print("done")

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s-seg summary (fused): 136 layers, 10,367,275 parameters, 0 gradients, 34.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1425.8±313.2 MB/s, size: 46.9 KB)
val: Scanning /kaggle/working/belt_damage/test/labels... 92 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 92/92 1.2Kit/s 0.1s
val: New cache created: /kaggle/working/belt_damage/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.0it/s 3.0s0.5ss
                   all         92        143      0.817      0.922      0.909      0.636      0.831      0.845      0.886      0.474
                  Tear         43         55      0.945          1      0.993      0.691      0.928      0.936      0.941      0.498
            patch work         37         42      0.941      0.952       0.96      0.717      0.922  

In [9]:
import os

ROOT = "/kaggle/working/belt-monitor"
FILES = {}

FILES["simulator/__init__.py"] = r''''''

FILES["simulator/config.py"] = r'''"""Every number the simulator uses, in one place.

The backend imports the belt geometry and VALID_RANGE from here too, so the
simulator and the backend can never disagree about them.
"""

# ---------------------------------------------------------------- belt
CONVEYOR_LENGTH_M = 3000          # distance from tail pulley to head pulley
BELT_LOOP_M = 6000                # the belt goes out and comes back
BELT_SPEED_MPS = 4.0              # design speed; one lap = 6000 / 4 = 1500 s = 25 min
BELT_WIDTH_MM = 1600

SENSOR_IDS = ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10"]
# S1 is 300 m from the tail pulley, S10 is at 3000 m, which is the head (drive) pulley.
SENSOR_POSITION_M = {"S1": 300, "S2": 600, "S3": 900, "S4": 1200, "S5": 1500,
                     "S6": 1800, "S7": 2100, "S8": 2400, "S9": 2700, "S10": 3000}

READING_INTERVAL_S = 60           # each sensor sends one message per minute

CHANNELS = ["temperature_c", "vibration_mm_s", "belt_speed_mps", "drift_mm"]

# ---------------------------------------------------------------- healthy signals
# Air temperature follows a daily wave: 36 C at 15:00, 20 C at 03:00.
AMBIENT_MEAN_C = 28.0
AMBIENT_SWING_C = 8.0
AMBIENT_HOTTEST_HOUR = 15

BEARING_RISE_C = 22.0             # a healthy roller bearing runs this much above the air
VIBRATION_BASE_MM_S = 2.0         # healthy vibration velocity (RMS)

# Random noise added to every reading (standard deviation).
NOISE_SD = {"temperature_c": 0.8, "vibration_mm_s": 0.25,
            "belt_speed_mps": 0.02, "drift_mm": 6.0}

# Each sensor point gets one fixed offset per channel, drawn once from
# -OFFSET_MAX..+OFFSET_MAX. No two sensor points read exactly the same.
OFFSET_MAX = {"temperature_c": 3.0, "vibration_mm_s": 0.3,
              "belt_speed_mps": 0.0, "drift_mm": 5.0}

# ---------------------------------------------------------------- failures
# A failure episode has three moments:
#   onset    -> the fault starts to develop
#   failure  -> the fault is fully developed (the values below are reached)
#   repaired -> maintenance has fixed it; readings go back to healthy
# Real faults take days. The durations are compressed to hours so that a
# 60-day run holds enough episodes to train on. The README must say so.
FAILURE_DURATION_H = {"seized_roller": (8, 24),   # (shortest, longest) onset -> failure
                      "belt_drift": (6, 18),
                      "belt_slip": (4, 12)}
FAILURE_MEAN_GAP_H = 36           # average quiet time between two episodes of one type
REPAIR_MINUTES = (30, 90)         # failure -> repaired

ROLLER_VIBRATION_ADD = 7.0        # mm/s added at failure. Grows as progress ** 1.5 (early)
ROLLER_TEMPERATURE_ADD = 50.0     # deg C added at failure. Grows as progress ** 3 (late)
DRIFT_ADD_MM = 150.0              # sideways shift at failure. Grows as progress ** 2
SLIP_SPEED_LOSS = 0.20            # share of speed lost at failure (4.0 -> 3.2 m/s)

# ---------------------------------------------------------------- messy data
P_DROPPED = 0.005                 # chance a message is never sent
P_NULL_FIELD = 0.003              # chance one value inside a message is null
P_SPIKE = 0.002                   # chance one value is replaced by a wild number
P_LATE = 0.01                     # chance a message arrives late
LATE_MINUTES = (2, 15)

OUTAGE_MEAN_GAP_H = 240           # per sensor: one silent period about every 10 days
OUTAGE_MINUTES = (10, 120)
STUCK_MEAN_GAP_H = 240            # per sensor: one frozen channel about every 10 days
STUCK_MINUTES = (30, 180)

# A value outside these limits cannot be a real measurement. Half of all
# spikes are pushed outside them, the other half stay inside.
VALID_RANGE = {"temperature_c": (-20.0, 200.0),
               "vibration_mm_s": (0.0, 50.0),
               "belt_speed_mps": (0.0, 8.0),
               "drift_mm": (-800.0, 800.0)}      # 800 mm = half the belt width

IMPOSSIBLE_SPIKES = {"temperature_c": [-273.0, 999.0],
                     "vibration_mm_s": [-5.0, 500.0],
                     "belt_speed_mps": [-3.0, 40.0],
                     "drift_mm": [-5000.0, 5000.0]}
# A spike that stays inside the valid range: the true value plus one of these.
PLAUSIBLE_SPIKE_ADD = {"temperature_c": (25.0, 60.0),
                       "vibration_mm_s": (5.0, 20.0),
                       "belt_speed_mps": (-2.5, -1.0),
                       "drift_mm": (100.0, 300.0)}

DECIMALS = {"temperature_c": 2, "vibration_mm_s": 3, "belt_speed_mps": 3, "drift_mm": 1}
'''

FILES["simulator/generate.py"] = r'''"""Conveyor belt sensor simulator.

One BeltSimulator object produces three things:
  ticks()      -> the sensor messages, minute by minute, in the order they ARRIVE
  failures     -> ground truth: every failure episode that was injected
  data_faults  -> ground truth: every piece of messy data that was injected

Time is counted in whole minutes from the start. "m" always means "minute number".
"""
import math
import random
from datetime import datetime, timedelta

from simulator import config as C


class BeltSimulator:
    def __init__(self, days, seed, start="2026-09-01T00:00:00Z", messy=True):
        self.rng = random.Random(seed)          # same seed -> exactly the same data
        self.total_min = int(days * 24 * 60)
        self.start = datetime.fromisoformat(start.replace("Z", "+00:00"))
        self.messy = messy                      # False -> no messy data, failures only
        self.data_faults = []

        # One fixed offset per sensor per channel.
        self.offsets = {}
        for sid in C.SENSOR_IDS:
            self.offsets[sid] = {}
            for ch in C.CHANNELS:
                self.offsets[sid][ch] = self.rng.uniform(-C.OFFSET_MAX[ch], C.OFFSET_MAX[ch])

        # Everything that has a start and an end is decided now, before any reading exists.
        self.failures = self._schedule_failures()
        self.outages = self._schedule_windows("outage", C.OUTAGE_MEAN_GAP_H, C.OUTAGE_MINUTES)
        self.stuck = self._schedule_windows("stuck", C.STUCK_MEAN_GAP_H, C.STUCK_MINUTES)

    # ------------------------------------------------------------ helpers
    def ts(self, m):
        """Minute number -> timestamp text, e.g. 2026-09-01T00:05:00Z."""
        return (self.start + timedelta(minutes=m)).strftime("%Y-%m-%dT%H:%M:%SZ")

    def ambient(self, m):
        """Air temperature at minute m: a daily wave, hottest at 15:00."""
        t = self.start + timedelta(minutes=m)
        hour = t.hour + t.minute / 60
        angle = 2 * math.pi * (hour - C.AMBIENT_HOTTEST_HOUR) / 24
        return C.AMBIENT_MEAN_C + C.AMBIENT_SWING_C * math.cos(angle)

    def _log_fault(self, fault_type, sensor_id, channel, start_m, end_m, detail=""):
        self.data_faults.append({"fault_type": fault_type, "sensor_id": sensor_id,
                                 "channel": channel, "start_ts": self.ts(start_m),
                                 "end_ts": self.ts(end_m), "detail": detail})

    # ------------------------------------------------------------ schedules
    def _schedule_failures(self):
        failures = []
        for ftype in C.FAILURE_DURATION_H:
            shortest_h, longest_h = C.FAILURE_DURATION_H[ftype]
            m = int(self.rng.expovariate(1 / (C.FAILURE_MEAN_GAP_H * 60)))
            while True:
                duration = self.rng.randint(shortest_h * 60, longest_h * 60)
                repair = self.rng.randint(C.REPAIR_MINUTES[0], C.REPAIR_MINUTES[1])
                if m + duration + repair >= self.total_min:
                    break
                # Slip happens at the drive pulley and slows the whole belt,
                # so it belongs to no single sensor.
                sensor_id = "ALL" if ftype == "belt_slip" else self.rng.choice(C.SENSOR_IDS)
                new = {"type": ftype, "sensor_id": sensor_id, "onset_min": m,
                       "failure_min": m + duration, "repaired_min": m + duration + repair,
                       "direction": self.rng.choice([-1, 1])}
                # Keep it only if this sensor has no other episode at the same time.
                clash = False
                for old in failures:
                    if (old["sensor_id"] == sensor_id and new["onset_min"] < old["repaired_min"]
                            and old["onset_min"] < new["repaired_min"]):
                        clash = True
                if not clash:
                    failures.append(new)
                gap = int(self.rng.expovariate(1 / (C.FAILURE_MEAN_GAP_H * 60)))
                m = m + duration + repair + gap

        failures.sort(key=lambda f: f["onset_min"])
        for number, f in enumerate(failures, start=1):
            f["failure_id"] = f"F{number:03d}"
            f["onset_ts"] = self.ts(f["onset_min"])
            f["failure_ts"] = self.ts(f["failure_min"])
            f["repaired_ts"] = self.ts(f["repaired_min"])
        return failures

    def _schedule_windows(self, kind, mean_gap_h, minutes_range):
        """Outage and stuck periods. Returns {sensor_id: [(start_m, end_m, channel), ...]}."""
        windows = {sid: [] for sid in C.SENSOR_IDS}
        if not self.messy:
            return windows
        for sid in C.SENSOR_IDS:
            m = int(self.rng.expovariate(1 / (mean_gap_h * 60)))
            while m < self.total_min:
                length = self.rng.randint(minutes_range[0], minutes_range[1])
                end_m = min(m + length, self.total_min - 1)
                channel = "all" if kind == "outage" else self.rng.choice(C.CHANNELS)
                windows[sid].append((m, end_m, channel))
                self._log_fault(kind, sid, channel, m, end_m)
                m = end_m + int(self.rng.expovariate(1 / (mean_gap_h * 60)))
        return windows

    @staticmethod
    def _window_at(windows, m):
        for w in windows:
            if w[0] <= m <= w[1]:
                return w
        return None

    # ------------------------------------------------------------ signal model
    def _healthy(self, sid, ambient):
        off = self.offsets[sid]
        g = self.rng.gauss
        return {
            "temperature_c": ambient + C.BEARING_RISE_C + off["temperature_c"]
                             + g(0, C.NOISE_SD["temperature_c"]),
            "vibration_mm_s": max(0.0, C.VIBRATION_BASE_MM_S + off["vibration_mm_s"]
                                  + g(0, C.NOISE_SD["vibration_mm_s"])),
            "belt_speed_mps": C.BELT_SPEED_MPS + g(0, C.NOISE_SD["belt_speed_mps"]),
            "drift_mm": off["drift_mm"] + g(0, C.NOISE_SD["drift_mm"]),
        }

    @staticmethod
    def progress(f, m):
        """0.0 at onset, 1.0 at failure, stays 1.0 until repaired."""
        p = (m - f["onset_min"]) / (f["failure_min"] - f["onset_min"])
        return min(1.0, p)

    def _apply_failure(self, values, f, m):
        p = self.progress(f, m)
        if f["type"] == "seized_roller":
            values["vibration_mm_s"] += C.ROLLER_VIBRATION_ADD * p ** 1.5
            values["temperature_c"] += C.ROLLER_TEMPERATURE_ADD * p ** 3
        elif f["type"] == "belt_drift":
            values["drift_mm"] += f["direction"] * C.DRIFT_ADD_MM * p ** 2
        elif f["type"] == "belt_slip":
            values["belt_speed_mps"] *= 1 - C.SLIP_SPEED_LOSS * p ** 2

    # ------------------------------------------------------------ main loop
    def ticks(self):
        """Yield (arrival_ts, [messages]) once per minute.

        A message that is late is held back and comes out in a later minute,
        so its own "ts" is older than the arrival_ts it comes out with.
        """
        rng = self.rng
        pending = {}        # arrival minute -> messages that will arrive then
        frozen = {}         # (sensor, stuck window start) -> the value it is stuck at

        for m in range(self.total_min):
            ambient = self.ambient(m)
            active = [f for f in self.failures if f["onset_min"] <= m < f["repaired_min"]]

            for sid in C.SENSOR_IDS:
                # 1. The true physical values.
                values = self._healthy(sid, ambient)
                for f in active:
                    if f["sensor_id"] == sid or f["sensor_id"] == "ALL":
                        self._apply_failure(values, f, m)
                for ch in C.CHANNELS:
                    values[ch] = round(values[ch], C.DECIMALS[ch])

                delay = 0
                if self.messy:
                    # 2. Outage: the sensor is silent, no message at all.
                    if self._window_at(self.outages[sid], m):
                        continue
                    # 3. Dropped: this single message is never sent.
                    if rng.random() < C.P_DROPPED:
                        self._log_fault("dropped", sid, "all", m, m)
                        continue
                    # 4. Stuck: one channel repeats the value it had when it froze.
                    w = self._window_at(self.stuck[sid], m)
                    if w:
                        key = (sid, w[0])
                        if key not in frozen:
                            frozen[key] = values[w[2]]
                        values[w[2]] = frozen[key]
                    # 5. Spike: one value replaced by a wild number.
                    if rng.random() < C.P_SPIKE:
                        ch = rng.choice(C.CHANNELS)
                        if rng.random() < 0.5:
                            values[ch] = rng.choice(C.IMPOSSIBLE_SPIKES[ch])
                            kind = "impossible"
                        else:
                            low, high = C.PLAUSIBLE_SPIKE_ADD[ch]
                            values[ch] = round(values[ch] + rng.uniform(low, high), C.DECIMALS[ch])
                            kind = "plausible"
                        self._log_fault("spike", sid, ch, m, m, f"{kind} value={values[ch]}")
                    # 6. Null field: the message arrives but one value is empty.
                    if rng.random() < C.P_NULL_FIELD:
                        ch = rng.choice(C.CHANNELS)
                        values[ch] = None
                        self._log_fault("null_field", sid, ch, m, m)
                    # 7. Late: the message is sent, but arrives some minutes later.
                    if rng.random() < C.P_LATE:
                        delay = rng.randint(C.LATE_MINUTES[0], C.LATE_MINUTES[1])
                        self._log_fault("late", sid, "all", m, m, f"delay_min={delay}")

                message = {"sensor_id": sid, "ts": self.ts(m)}
                message.update(values)
                pending.setdefault(m + delay, []).append(message)

            yield self.ts(m), pending.pop(m, [])

        # Messages that were still on their way when the run ended.
        for m in sorted(pending):
            yield self.ts(m), pending[m]
'''

FILES["simulator/run.py"] = r'''"""Command line for the simulator.

History for training (writes three CSV files):
    python -m simulator.run batch --days 60 --seed 42 --out data/train

Live feed for the backend (one HTTP POST per simulated minute):
    python -m simulator.run live --url http://127.0.0.1:8000/readings --seed 7 --speedup 600
"""
import argparse
import csv
import json
import os
import time
import urllib.error
import urllib.request

from simulator import config as C
from simulator.generate import BeltSimulator

FAILURE_COLUMNS = ["failure_id", "type", "sensor_id", "onset_ts", "failure_ts", "repaired_ts"]
FAULT_COLUMNS = ["fault_type", "sensor_id", "channel", "start_ts", "end_ts", "detail"]


def write_csv(path, columns, rows):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def run_batch(args):
    sim = BeltSimulator(args.days, args.seed, args.start, messy=not args.clean)
    os.makedirs(args.out, exist_ok=True)
    count = 0
    with open(os.path.join(args.out, "readings.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["arrival_ts", "sensor_id", "ts"] + C.CHANNELS)
        for arrival_ts, messages in sim.ticks():
            for msg in messages:
                # A null value is written as an empty cell.
                row = [arrival_ts, msg["sensor_id"], msg["ts"]]
                for ch in C.CHANNELS:
                    row.append("" if msg[ch] is None else msg[ch])
                writer.writerow(row)
                count += 1
    write_csv(os.path.join(args.out, "failures.csv"), FAILURE_COLUMNS, sim.failures)
    write_csv(os.path.join(args.out, "data_faults.csv"), FAULT_COLUMNS, sim.data_faults)

    expected = sim.total_min * len(C.SENSOR_IDS)
    print(f"readings written : {count} of {expected} possible")
    for ftype in C.FAILURE_DURATION_H:
        print(f"{ftype:17}: {sum(1 for x in sim.failures if x['type'] == ftype)} episodes")
    for kind in ["outage", "stuck", "spike", "null_field", "dropped", "late"]:
        print(f"{kind:17}: {sum(1 for x in sim.data_faults if x['fault_type'] == kind)}")


def post_json(url, payload):
    body = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=body,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=5) as response:
        return response.status


def run_live(args):
    sim = BeltSimulator(args.days, args.seed, args.start, messy=not args.clean)
    os.makedirs(args.out, exist_ok=True)
    # The failure schedule is known before the first reading, so write it now.
    write_csv(os.path.join(args.out, "failures.csv"), FAILURE_COLUMNS, sim.failures)
    pause = C.READING_INTERVAL_S / args.speedup      # real seconds per simulated minute
    sent = failed = 0
    try:
        for arrival_ts, messages in sim.ticks():
            if messages:
                try:
                    post_json(args.url, messages)
                    sent += len(messages)
                except (urllib.error.URLError, OSError) as error:
                    failed += len(messages)
                    print(f"{arrival_ts} POST failed: {error}")
            time.sleep(pause)
    except KeyboardInterrupt:
        print("stopped by user")
    finally:
        write_csv(os.path.join(args.out, "data_faults.csv"), FAULT_COLUMNS, sim.data_faults)
        print(f"messages sent: {sent}, not delivered: {failed}")


def main():
    parser = argparse.ArgumentParser(description="Conveyor belt sensor simulator")
    sub = parser.add_subparsers(dest="mode", required=True)
    for name in ["batch", "live"]:
        p = sub.add_parser(name)
        p.add_argument("--days", type=float, default=60)
        p.add_argument("--seed", type=int, default=42)
        p.add_argument("--start", default="2026-09-01T00:00:00Z")
        p.add_argument("--clean", action="store_true", help="no messy data, failures only")
        p.add_argument("--out", default="data/train" if name == "batch" else "data/live")
        if name == "live":
            p.add_argument("--url", default="http://127.0.0.1:8000/readings")
            p.add_argument("--speedup", type=float, default=600,
                           help="600 means one simulated minute takes 0.1 real seconds")
    args = parser.parse_args()
    if args.mode == "batch":
        run_batch(args)
    else:
        run_live(args)


if __name__ == "__main__":
    main()
'''

FILES["backend/__init__.py"] = r''''''

FILES["backend/db.py"] = r'''"""The database: one SQLite file with four tables."""
import os
import sqlite3

# The file sits in belt-monitor/data/belt.db unless BELT_DB says otherwise.
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DB_PATH = os.environ.get("BELT_DB") or os.path.join(PROJECT_DIR, "data", "belt.db")

SCHEMA = """
CREATE TABLE IF NOT EXISTS readings (
    sensor_id       TEXT NOT NULL,
    ts              TEXT NOT NULL,      -- when the sensor measured
    temperature_c   REAL,               -- NULL means missing or impossible
    vibration_mm_s  REAL,
    belt_speed_mps  REAL,
    drift_mm        REAL,
    flags           TEXT NOT NULL,      -- what was wrong with this reading, '' if nothing
    PRIMARY KEY (sensor_id, ts)         -- the same reading cannot be stored twice
);
CREATE INDEX IF NOT EXISTS readings_by_ts ON readings (ts);

CREATE TABLE IF NOT EXISTS rejected (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    reason  TEXT NOT NULL,
    raw     TEXT NOT NULL               -- the message exactly as it came in
);

CREATE TABLE IF NOT EXISTS anomalies (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id   TEXT NOT NULL,          -- 'ALL' for belt slip
    position_m  INTEGER NOT NULL,
    type        TEXT NOT NULL,
    severity    TEXT NOT NULL,          -- 'warning' or 'critical'
    value       REAL,                   -- the latest value that broke the limit
    started_ts  TEXT NOT NULL,
    last_ts     TEXT NOT NULL,
    ended_ts    TEXT                    -- NULL while the anomaly is still active
);

CREATE TABLE IF NOT EXISTS predictions (
    id                    INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id             TEXT NOT NULL,     -- 'ALL' for belt slip
    position_m            INTEGER NOT NULL,
    failure_type          TEXT NOT NULL,     -- what the model expects to fail
    probability           REAL NOT NULL,     -- how sure the model is, 0 to 1
    hours_left            REAL NOT NULL,     -- the model's estimate at last_ts
    predicted_failure_ts  TEXT NOT NULL,     -- last_ts + hours_left
    started_ts            TEXT NOT NULL,
    last_ts               TEXT NOT NULL,
    ended_ts              TEXT               -- NULL while the prediction is still active
);
"""


def connect(path=None):
    path = path or DB_PATH
    os.makedirs(os.path.dirname(path), exist_ok=True)
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row          # rows can be read by column name
    conn.execute("PRAGMA journal_mode=WAL")  # the dashboard can read while the backend writes
    conn.executescript(SCHEMA)
    return conn
'''

FILES["backend/validate.py"] = r'''"""Check one incoming message. Nothing is ever guessed or repaired here."""
from datetime import datetime

from simulator import config as C

TS_FORMAT = "%Y-%m-%dT%H:%M:%SZ"


def validate(message):
    """Return (row, flags, reason).

    reason is None  -> the message is usable. row holds the cleaned values,
                       flags lists what was wrong with single values.
    reason is text  -> the whole message is rejected, row is None.
    """
    if not isinstance(message, dict):
        return None, [], "not a JSON object"
    if message.get("sensor_id") not in C.SENSOR_IDS:
        return None, [], "unknown sensor_id"
    try:
        datetime.strptime(message.get("ts"), TS_FORMAT)
    except (TypeError, ValueError):
        return None, [], "bad timestamp"

    row = {"sensor_id": message["sensor_id"], "ts": message["ts"]}
    flags = []
    for ch in C.CHANNELS:
        value = message.get(ch)
        low, high = C.VALID_RANGE[ch]
        if value is None:
            row[ch] = None
            flags.append("missing:" + ch)
        elif isinstance(value, bool) or not isinstance(value, (int, float)) or value != value:
            row[ch] = None                       # text, true/false or NaN
            flags.append("not_a_number:" + ch)
        elif value < low or value > high:
            row[ch] = None                       # cannot be a real measurement
            flags.append("out_of_range:" + ch)
        else:
            row[ch] = float(value)
    return row, flags, None
'''

FILES["backend/analyse.py"] = r'''"""Turn stored readings into anomalies.

A limit must be broken by 3 readings in a row before an anomaly opens, and
3 readings in a row must be back inside the limit before it closes.
One wild reading therefore changes nothing.
"""
from datetime import datetime
from statistics import median

from simulator import config as C
from backend.validate import TS_FORMAT

IN_A_ROW = 3
STUCK_AFTER = 5              # the same value this many times in a row = stuck sensor
OFFLINE_AFTER_MIN = 5        # no reading for this many minutes = sensor offline

# anomaly type: (channel, warning limit, critical limit)
LIMITS = {
    "high_temperature": ("temperature_c", 70.0, 85.0),
    "high_vibration": ("vibration_mm_s", 4.5, 7.1),
    "belt_drift": ("drift_mm", 75.0, 120.0),          # distance from centre, either side
}
SLIP_WARNING = 0.95 * C.BELT_SPEED_MPS                 # 3.8 m/s
SLIP_CRITICAL = 0.85 * C.BELT_SPEED_MPS                # 3.4 m/s
DRIVE_PULLEY_M = C.CONVEYOR_LENGTH_M


def level(values, warning, critical, low_is_bad=False):
    """Judge the newest 3 values. Returns 'critical', 'warning', 'normal' or 'unknown'."""
    if len(values) < IN_A_ROW or None in values:
        return "unknown"                                # not enough trusted readings
    if low_is_bad:
        values = [-v for v in values]
        warning, critical = -warning, -critical
    if all(v > critical for v in values):
        return "critical"
    if all(v > warning for v in values):
        return "warning"
    if all(v <= warning for v in values):
        return "normal"
    return "unknown"                                    # mixed: keep the current state


def set_anomaly(conn, active, sensor_id, position_m, kind, verdict, ts, value):
    """Open, update or close the anomaly (sensor_id, kind) according to the verdict."""
    key = (sensor_id, kind)
    if verdict == "unknown":
        return
    if verdict == "normal":
        if key in active:
            conn.execute("UPDATE anomalies SET ended_ts = ? WHERE id = ?", (ts, active[key]))
            del active[key]
        return
    if key in active:
        conn.execute("UPDATE anomalies SET severity = ?, value = ?, last_ts = ? WHERE id = ?",
                     (verdict, value, ts, active[key]))
    else:
        cur = conn.execute(
            "INSERT INTO anomalies (sensor_id, position_m, type, severity, value, started_ts, last_ts)"
            " VALUES (?, ?, ?, ?, ?, ?, ?)", (sensor_id, position_m, kind, verdict, value, ts, ts))
        active[key] = cur.lastrowid


def analyse(conn, sensor_ids):
    """Run every rule. sensor_ids = the sensors that just sent a reading."""
    active = {(r["sensor_id"], r["type"]): r["id"] for r in
              conn.execute("SELECT id, sensor_id, type FROM anomalies WHERE ended_ts IS NULL")}
    clock = conn.execute("SELECT MAX(ts) FROM readings").fetchone()[0]
    if clock is None:
        return

    for sid in C.SENSOR_IDS:
        position = C.SENSOR_POSITION_M[sid]
        newest = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT ?",
                              (sid, STUCK_AFTER)).fetchall()
        if not newest:
            continue

        # Rule 1: sensor offline. Its newest reading is too far behind the clock.
        behind = datetime.strptime(clock, TS_FORMAT) - datetime.strptime(newest[0]["ts"], TS_FORMAT)
        minutes = behind.total_seconds() / 60
        verdict = "warning" if minutes > OFFLINE_AFTER_MIN else "normal"
        set_anomaly(conn, active, sid, position, "sensor_offline", verdict, clock, minutes)
        if sid not in sensor_ids:
            continue                                    # nothing new from this sensor

        # Rule 2: stuck sensor. One channel shows the exact same value 5 times in a row.
        stuck = []
        for ch in C.CHANNELS:
            values = [r[ch] for r in newest]
            if len(values) == STUCK_AFTER and values[0] is not None and len(set(values)) == 1:
                stuck.append(ch)
        if stuck:
            flags = newest[0]["flags"].split(",") if newest[0]["flags"] else []
            flags += ["stuck:" + ch for ch in stuck if "stuck:" + ch not in flags]
            conn.execute("UPDATE readings SET flags = ? WHERE sensor_id = ? AND ts = ?",
                         (",".join(flags), sid, newest[0]["ts"]))
        set_anomaly(conn, active, sid, position, "sensor_stuck",
                    "warning" if stuck else "normal", newest[0]["ts"], None)

        # Rule 3: limits on temperature, vibration and drift.
        for kind, (ch, warning, critical) in LIMITS.items():
            values = []
            for r in newest[:IN_A_ROW]:
                trusted = r[ch] is not None and "stuck:" + ch not in r["flags"] and ch not in stuck
                values.append(abs(r[ch]) if trusted else None)
            set_anomaly(conn, active, sid, position, kind, level(values, warning, critical),
                        newest[0]["ts"], values[0])

    # Rule 4: belt slip. The belt is one piece, so its speed is judged once per minute
    # as the middle value (median) of all sensors, not sensor by sensor.
    rows = conn.execute("SELECT ts, belt_speed_mps FROM readings WHERE ts IN "
                        "(SELECT DISTINCT ts FROM readings ORDER BY ts DESC LIMIT ?)",
                        (IN_A_ROW,)).fetchall()
    by_minute = {}
    for r in rows:
        by_minute.setdefault(r["ts"], [])
        if r["belt_speed_mps"] is not None:
            by_minute[r["ts"]].append(r["belt_speed_mps"])
    speeds = []
    for ts in sorted(by_minute, reverse=True):
        speeds.append(median(by_minute[ts]) if len(by_minute[ts]) >= 5 else None)
    set_anomaly(conn, active, "ALL", DRIVE_PULLEY_M, "belt_slip",
                level(speeds, SLIP_WARNING, SLIP_CRITICAL, low_is_bad=True), clock,
                speeds[0] if speeds else None)
'''

FILES["backend/ingest.py"] = r'''"""Receive one batch of messages: validate, store, analyse."""
import json

from backend.analyse import analyse
from backend.validate import validate


def ingest(conn, messages, commit=True, with_predictions=False):
    """Store a list of messages. Returns counts of what happened to them."""
    summary = {"received": len(messages), "stored": 0, "rejected": 0, "duplicate": 0, "late": 0}
    # The backend's clock is the newest timestamp it has stored so far.
    clock = conn.execute("SELECT MAX(ts) FROM readings").fetchone()[0]
    touched = set()

    for message in messages:
        row, flags, reason = validate(message)
        if reason is not None:
            conn.execute("INSERT INTO rejected (reason, raw) VALUES (?, ?)",
                         (reason, json.dumps(message, default=str)))
            summary["rejected"] += 1
            continue
        late = clock is not None and row["ts"] < clock
        if late:
            flags.append("late")                 # older than something already stored
        cur = conn.execute(
            "INSERT OR IGNORE INTO readings VALUES (?, ?, ?, ?, ?, ?, ?)",
            (row["sensor_id"], row["ts"], row["temperature_c"], row["vibration_mm_s"],
             row["belt_speed_mps"], row["drift_mm"], ",".join(flags)))
        if cur.rowcount == 0:
            summary["duplicate"] += 1            # this sensor and timestamp are already stored
            continue
        summary["stored"] += 1
        summary["late"] += 1 if late else 0
        touched.add(row["sensor_id"])

    if summary["stored"]:
        analyse(conn, touched)
        # Every 5 simulated minutes, ask the prediction model.
        newest = conn.execute("SELECT MAX(ts) FROM readings").fetchone()[0]
        if with_predictions and newest != clock and int(newest[14:16]) % 5 == 0:
            from backend.predict import predict
            predict(conn, newest)
    if commit:
        conn.commit()
    return summary
'''

FILES["backend/predict.py"] = r'''"""Run the trained model on the newest readings and keep the predictions table up to date.

Called by ingest() every 5 simulated minutes. If no trained model file exists yet,
it does nothing, so the backend also works before the model is trained.
"""
import os
from datetime import datetime, timedelta

from simulator import config as C
from backend.db import PROJECT_DIR
from backend.validate import TS_FORMAT

MODEL_PATH = os.environ.get("BELT_MODEL") or os.path.join(PROJECT_DIR, "model", "belt_model.joblib")
SLIP_SENSORS_NEEDED = 6      # belt slip is predicted only when this many sensors agree
_model = None


def load_model():
    global _model
    if _model is None and os.path.exists(MODEL_PATH):
        import joblib
        _model = joblib.load(MODEL_PATH)
    return _model


def set_prediction(conn, active, sensor_id, position_m, kind, verdict, ts, probability, hours_left):
    """Open, update or close the prediction (sensor_id, kind). verdict: True, False or None."""
    key = (sensor_id, kind)
    if verdict is None:
        return                                             # no clear answer: keep the current state
    if verdict is False:
        if key in active:
            conn.execute("UPDATE predictions SET ended_ts = ? WHERE id = ?", (ts, active[key]))
        return
    fails_at = (datetime.strptime(ts, TS_FORMAT) + timedelta(hours=hours_left)).strftime(TS_FORMAT)
    if key in active:
        conn.execute("UPDATE predictions SET probability = ?, hours_left = ?, predicted_failure_ts = ?,"
                     " last_ts = ? WHERE id = ?", (probability, hours_left, fails_at, ts, active[key]))
    else:
        conn.execute("INSERT INTO predictions (sensor_id, position_m, failure_type, probability,"
                     " hours_left, predicted_failure_ts, started_ts, last_ts) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                     (sensor_id, position_m, kind, probability, hours_left, fails_at, ts, ts))


def predict(conn, clock):
    """clock: the newest timestamp in the database (text)."""
    model = load_model()
    if model is None:
        return
    import pandas as pd
    from model.features import FEATURES, HISTORY_MIN, build_features

    step, in_a_row = model["step_min"], model["alert_in_a_row"]
    now = datetime.strptime(clock, TS_FORMAT)
    since = (now - timedelta(minutes=HISTORY_MIN + step * in_a_row)).strftime(TS_FORMAT)
    readings = pd.read_sql("SELECT * FROM readings WHERE ts >= ?", conn, params=(since,))
    rows = build_features(readings)
    # Keep the newest 3 prediction moments: now, 5 minutes ago, 10 minutes ago.
    moments = [pd.Timestamp(now - timedelta(minutes=step * k)) for k in range(in_a_row)]
    rows = rows[rows["ts"].dt.tz_localize(None).isin(moments)]
    if rows.empty:
        return
    probabilities = model["classifier"].predict_proba(rows[FEATURES])
    classes = model["classifier"].classes_
    rows = rows.assign(predicted=classes[probabilities.argmax(axis=1)],
                       probability=probabilities.max(axis=1))

    active = {(r["sensor_id"], r["failure_type"]): r["id"] for r in
              conn.execute("SELECT id, sensor_id, failure_type FROM predictions WHERE ended_ts IS NULL")}
    slip_votes = []
    for sid in C.SENSOR_IDS:
        mine = rows[rows["sensor_id"] == sid].sort_values("ts")
        if len(mine) < in_a_row:
            continue
        newest = mine.iloc[[-1]]
        answers = list(mine["predicted"])
        sure = all(mine["probability"] >= model["alert_probability"])
        for kind in ["seized_roller", "belt_drift", "belt_slip"]:
            if all(a == kind for a in answers) and sure:
                verdict = True                             # 3 sure answers in a row
            elif all(a != kind for a in answers):
                verdict = False                            # 3 answers in a row say it is not this
            else:
                verdict = None
            hours = float(model["regressors"][kind].predict(newest[FEATURES])[0]) if verdict else 0.0
            hours = max(hours, 0.0)
            if kind == "belt_slip":
                slip_votes.append((verdict, float(newest["probability"].iloc[0]), hours))
            else:
                set_prediction(conn, active, sid, C.SENSOR_POSITION_M[sid], kind, verdict, clock,
                               float(newest["probability"].iloc[0]), hours)

    # Belt slip concerns the whole belt, so the sensors vote.
    yes = [v for v in slip_votes if v[0] is True]
    no = [v for v in slip_votes if v[0] is False]
    if len(yes) >= SLIP_SENSORS_NEEDED:
        yes.sort(key=lambda v: v[2])
        middle = yes[len(yes) // 2]
        set_prediction(conn, active, "ALL", C.CONVEYOR_LENGTH_M, "belt_slip", True, clock, middle[1], middle[2])
    elif len(no) >= SLIP_SENSORS_NEEDED:
        set_prediction(conn, active, "ALL", C.CONVEYOR_LENGTH_M, "belt_slip", False, clock, 0.0, 0.0)
'''

FILES["backend/app.py"] = r'''"""The HTTP layer. Each function below answers one URL.

Start it from the belt-monitor folder with:
    python -m uvicorn backend.app:app --port 8000
"""
from fastapi import Body, FastAPI

from simulator import config as C
from backend.db import connect
from backend.ingest import ingest

app = FastAPI(title="Belt monitor")

PROCESS_ANOMALIES = ["high_temperature", "high_vibration", "belt_drift"]
SENSOR_FAULTS = ["sensor_offline", "sensor_stuck"]


@app.get("/health")
def health():
    conn = connect()
    count = conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
    clock = conn.execute("SELECT MAX(ts) FROM readings").fetchone()[0]
    conn.close()
    return {"status": "ok", "readings": count, "clock": clock}


@app.post("/readings")
def post_readings(messages: list[dict] = Body(...)):
    """The simulator sends its messages here. Each message is checked one by one,
    so one bad message cannot make the whole batch fail."""
    conn = connect()
    summary = ingest(conn, messages, with_predictions=True)
    conn.close()
    return summary


@app.get("/readings")
def get_readings(sensor_id: str, limit: int = 240):
    """The newest readings of one sensor, oldest first (for the dashboard charts)."""
    conn = connect()
    rows = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT ?",
                        (sensor_id, limit)).fetchall()
    conn.close()
    return [dict(r) for r in reversed(rows)]


@app.get("/anomalies")
def get_anomalies(active: bool = True, limit: int = 200):
    """Active anomalies, or the most recent ones of any state when active=false."""
    conn = connect()
    where = "WHERE ended_ts IS NULL" if active else ""
    rows = conn.execute(f"SELECT * FROM anomalies {where} ORDER BY started_ts DESC LIMIT ?",
                        (limit,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]


@app.get("/predictions")
def get_predictions(active: bool = True, limit: int = 200):
    """What the model expects to fail, where, and when."""
    conn = connect()
    where = "WHERE ended_ts IS NULL" if active else ""
    rows = conn.execute(f"SELECT * FROM predictions {where} ORDER BY started_ts DESC LIMIT ?",
                        (limit,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]


@app.get("/sensors")
def get_sensors():
    """One entry per sensor point: where it is, its newest reading and its status."""
    conn = connect()
    active = conn.execute("SELECT sensor_id, type, severity FROM anomalies "
                          "WHERE ended_ts IS NULL").fetchall()
    slip = [a for a in active if a["type"] == "belt_slip"]
    predicted = conn.execute("SELECT sensor_id, failure_type, hours_left FROM predictions "
                             "WHERE ended_ts IS NULL").fetchall()
    result = []
    for sid in C.SENSOR_IDS:
        newest = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT 1",
                              (sid,)).fetchone()
        mine = [a for a in active if a["sensor_id"] == sid]
        process = [a["type"] for a in mine if a["type"] in PROCESS_ANOMALIES]
        faults = [a["type"] for a in mine if a["type"] in SENSOR_FAULTS]
        if slip:
            process.append("belt_slip")        # slip slows the belt at every sensor point
        expected = [dict(p) for p in predicted if p["sensor_id"] in (sid, "ALL")]
        if newest is None or "sensor_offline" in faults:
            status = "no_data"                 # no current reading: it cannot be called normal
        elif process:
            status = "abnormal"                # a current reading is past its limit
        elif expected:
            status = "predicted_failure"       # inside the limits, but the model expects a failure
        else:
            status = "normal"
        result.append({"sensor_id": sid, "position_m": C.SENSOR_POSITION_M[sid],
                       "status": status, "anomalies": process, "sensor_faults": faults,
                       "predictions": expected, "newest": dict(newest) if newest else None})
    conn.close()
    return result
'''

FILES["backend/load_csv.py"] = r'''"""Load a readings.csv written by the simulator into the database.

    python -m backend.load_csv data/train/readings.csv

Rows are fed to ingest() minute by minute in arrival order, so the backend
sees exactly what it would see from the live feed.
"""
import csv
import sys
import time

from simulator import config as C
from backend.db import connect
from backend.ingest import ingest


def to_message(row):
    message = {"sensor_id": row["sensor_id"], "ts": row["ts"]}
    for ch in C.CHANNELS:
        message[ch] = None if row[ch] == "" else float(row[ch])
    return message


def load(csv_path, db_path=None):
    conn = connect(db_path)
    total = {"received": 0, "stored": 0, "rejected": 0, "duplicate": 0, "late": 0}
    batch, batch_arrival, batches = [], None, 0
    started = time.time()

    def flush():
        result = ingest(conn, batch, commit=False)
        for key in total:
            total[key] += result[key]

    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            if row["arrival_ts"] != batch_arrival and batch:
                flush()
                batch = []
                batches += 1
                if batches % 1000 == 0:
                    conn.commit()
            batch_arrival = row["arrival_ts"]
            batch.append(to_message(row))
    if batch:
        flush()
    conn.commit()
    print(f"{total} in {time.time() - started:.0f} s")
    return conn


if __name__ == "__main__":
    load(sys.argv[1], sys.argv[2] if len(sys.argv) > 2 else None)
'''

FILES["model/__init__.py"] = r''''''

FILES["model/features.py"] = r'''"""Turn raw readings into the numbers the prediction model looks at.

The same function is used for training (60 days at once) and for live
prediction (the last 27 hours), so both see features built the same way.
"""
import pandas as pd

CHANNELS = ["temperature_c", "vibration_mm_s", "belt_speed_mps", "drift_mm"]
BASELINE_MIN = 1440          # a sensor's own normal level = middle value of its last 24 hours
HISTORY_MIN = BASELINE_MIN + 200   # minutes of history needed to build one row of features

FEATURES = ["temp_level", "temp_change_1h", "temp_change_3h",
            "vib_level", "vib_change_1h", "vib_change_3h",
            "drift_level", "drift_change_1h", "drift_change_3h",
            "speed_level", "speed_change_1h", "speed_change_3h"]


def build_features(readings):
    """readings: DataFrame with columns sensor_id, ts, the four channels and flags.

    Returns one row per sensor per minute with the 12 FEATURES columns.
    A value that is missing, impossible or stuck is simply absent: the
    features are built from whatever trusted values remain.
    """
    df = readings.copy()
    df["ts"] = pd.to_datetime(df["ts"])
    # A channel that the backend flagged as stuck is not trusted.
    for ch in CHANNELS:
        stuck = df["flags"].str.contains("stuck:" + ch, regex=False)
        df.loc[stuck, ch] = float("nan")

    # One table per channel: rows = minutes, columns = sensors.
    minutes = pd.date_range(df["ts"].min(), df["ts"].max(), freq="min")
    wide = {}
    for ch in CHANNELS:
        table = df.pivot_table(index="ts", columns="sensor_id", values=ch, aggfunc="last")
        wide[ch] = table.reindex(minutes)
    sensors = list(wide["vibration_mm_s"].columns)

    # Temperature compared with the middle sensor of the same minute.
    # This removes the daily heat wave, which all sensors share.
    temp = wide["temperature_c"].sub(wide["temperature_c"].median(axis=1), axis=0)
    # The belt is one piece: its speed is the middle value of all sensors.
    belt_speed = wide["belt_speed_mps"].median(axis=1)
    speed = pd.DataFrame({sid: belt_speed for sid in sensors})

    tables = {}
    for name, table in [("temp", temp), ("vib", wide["vibration_mm_s"]),
                        ("drift", wide["drift_mm"]), ("speed", speed)]:
        # Middle value of the last 15 minutes: one wild reading cannot move it.
        now = table.rolling(15, min_periods=5).median()
        if name == "speed":
            level = now                                      # the normal belt speed is known: 4 m/s
        else:
            # Compare each sensor with its own normal level, so a sensor that
            # always reads a little high is not mistaken for a failing one.
            level = now - table.rolling(BASELINE_MIN, min_periods=180).median()
        if name == "drift":
            level, now = level.abs(), now.abs()              # left or right does not matter
        tables[name + "_level"] = level
        tables[name + "_change_1h"] = now - now.shift(60)    # compared with 1 hour earlier
        tables[name + "_change_3h"] = now - now.shift(180)   # compared with 3 hours earlier

    frames = []
    for sid in sensors:
        frame = pd.DataFrame({name: table[sid] for name, table in tables.items()})
        frame.insert(0, "sensor_id", sid)
        frame.insert(1, "ts", frame.index)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)[["sensor_id", "ts"] + FEATURES]
'''

FILES["model/train.py"] = r'''"""Train and test the failure prediction model.

    python -m model.train data/train.db data/train/failures.csv

It answers two questions for every sensor, every 5 minutes:
  1. Is a failure developing, and which one?      (classifier)
  2. If yes, how many hours are left?             (one regressor per failure type)

The history is cut in two by time. The model learns from the first part and is
tested on the second part, so no failure episode is seen in both.
"""
import sqlite3
import sys

import joblib
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

from model.features import FEATURES, build_features

STEP_MIN = 5                 # one prediction every 5 minutes
ALERT_PROBABILITY = 0.8      # how sure the model must be
ALERT_IN_A_ROW = 3           # for how many predictions in a row (3 x 5 = 15 minutes)
TRAIN_SHARE = 0.7
SETTLE_MIN = 20              # minutes after a repair that are left out of training and testing
TYPES = ["seized_roller", "belt_drift", "belt_slip"]
# Which backend anomalies belong to which failure type (used only for the comparison).
LIMIT_TYPES = {"seized_roller": ["high_vibration", "high_temperature"],
               "belt_drift": ["belt_drift"], "belt_slip": ["belt_slip"]}


def add_labels(rows, failures):
    """Give every row a label: 'none', a failure type, or 'settling' (just after a repair)."""
    rows = rows.copy()
    rows["label"] = "none"
    rows["hours_left"] = float("nan")
    rows["failure_id"] = ""
    # Slip first, so a fault on one sensor (roller, drift) overwrites it for that sensor.
    for f in failures.sort_values("type", key=lambda s: s != "belt_slip").itertuples():
        this_sensor = (rows["sensor_id"] == f.sensor_id) | (f.sensor_id == "ALL")
        # From onset to repair the fault is present. Before the failure moment the
        # hours left count down; from the failure moment on they are 0.
        present = this_sensor & (rows["ts"] >= f.onset_ts) & (rows["ts"] < f.repaired_ts)
        left = (f.failure_ts - rows.loc[present, "ts"]).dt.total_seconds() / 3600
        rows.loc[present, "label"] = f.type
        rows.loc[present, "hours_left"] = left.clip(lower=0)
        rows.loc[present, "failure_id"] = f.failure_id
        # Just after the repair the 15-minute window still holds failed readings.
        settled_ts = f.repaired_ts + pd.Timedelta(minutes=SETTLE_MIN)
        settling = this_sensor & (rows["ts"] >= f.repaired_ts) & (rows["ts"] < settled_ts)
        rows.loc[settling, "label"] = "settling"
    return rows


def find_alerts(rows, classifier):
    """Add the model's answer to every row, and mark where an alert would be raised."""
    rows = rows.sort_values(["sensor_id", "ts"]).copy()
    probabilities = classifier.predict_proba(rows[FEATURES])
    rows["predicted"] = classifier.classes_[probabilities.argmax(axis=1)]
    rows["probability"] = probabilities.max(axis=1)
    sure = (rows["predicted"] != "none") & (rows["probability"] >= ALERT_PROBABILITY)
    # An alert needs the same sure answer 3 times in a row on the same sensor.
    run = sure.astype(int)
    for back in range(1, ALERT_IN_A_ROW):
        same = (rows["sensor_id"] == rows["sensor_id"].shift(back)) & \
               (rows["predicted"] == rows["predicted"].shift(back))
        run = run + (sure.shift(back, fill_value=False) & same).astype(int)
    rows["alert"] = run == ALERT_IN_A_ROW
    return rows


def main(db_path, failures_path, model_path="model/belt_model.joblib"):
    conn = sqlite3.connect(db_path)
    readings = pd.read_sql("SELECT * FROM readings", conn)
    anomalies = pd.read_sql("SELECT * FROM anomalies", conn)
    anomalies["started_ts"] = pd.to_datetime(anomalies["started_ts"]).dt.tz_localize(None)
    failures = pd.read_csv(failures_path)
    for column in ["onset_ts", "failure_ts", "repaired_ts"]:
        failures[column] = pd.to_datetime(failures[column]).dt.tz_localize(None)

    rows = build_features(readings)
    rows["ts"] = rows["ts"].dt.tz_localize(None)
    rows = rows[rows["ts"].dt.minute % STEP_MIN == 0]
    rows = add_labels(rows, failures)

    # Cut the history by time, at a moment when no failure episode is running.
    start, end = rows["ts"].min(), rows["ts"].max()
    cut = start + (end - start) * TRAIN_SHARE
    while ((failures["onset_ts"] <= cut) & (failures["repaired_ts"] >= cut)).any():
        cut += pd.Timedelta(hours=1)
    usable = rows[rows["label"] != "settling"]
    train, test = usable[usable["ts"] < cut], usable[usable["ts"] >= cut]
    print(f"history cut at {cut}")
    print(f"train rows {len(train)}, test rows {len(test)}")
    for t in TYPES:
        print(f"  {t:14} episodes: train {train.loc[train.label == t, 'failure_id'].nunique()}, "
              f"test {test.loc[test.label == t, 'failure_id'].nunique()}")

    classifier = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=0)
    classifier.fit(train[FEATURES], train["label"])
    regressors = {}
    for t in TYPES:
        part = train[train["label"] == t]
        regressors[t] = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=0)
        regressors[t].fit(part[FEATURES], part["hours_left"])

    # ---------------- test on the part of the history the model has never seen
    test = find_alerts(test, classifier)
    print("\nTEST RESULTS")
    print(f"{'failure type':15}{'episodes':>9}{'warned':>8}{'model warning h':>16}{'fixed limits h':>17}"
          f"{'hours-left error h':>20}")
    test_failures = failures[failures["onset_ts"] >= cut]
    for t in TYPES:
        leads, limit_leads, errors = [], [], []
        episodes = test_failures[test_failures["type"] == t]
        for f in episodes.itertuples():
            # Alerts of the right type, raised before the failure moment.
            hit = test[(test["failure_id"] == f.failure_id) & test["alert"] &
                       (test["predicted"] == t) & (test["ts"] < f.failure_ts)]
            if len(hit):
                first = hit.sort_values("ts").iloc[0]
                leads.append((f.failure_ts - first["ts"]).total_seconds() / 3600)
                guess = regressors[t].predict(hit[FEATURES])
                errors.extend(abs(guess - hit["hours_left"]))
            # For comparison: when did the backend's fixed limits first react?
            mine = anomalies[(anomalies["sensor_id"] == f.sensor_id) & anomalies["type"].isin(LIMIT_TYPES[t]) &
                             (anomalies["started_ts"] >= f.onset_ts) & (anomalies["started_ts"] < f.failure_ts)]
            if len(mine):
                limit_leads.append((f.failure_ts - mine["started_ts"].min()).total_seconds() / 3600)
        leads = pd.Series(leads, dtype=float)
        errors = pd.Series(errors, dtype=float)
        limit_leads = pd.Series(limit_leads, dtype=float)
        print(f"{t:15}{len(episodes):>9}{len(leads):>8}{leads.median():>16.1f}{limit_leads.median():>17.1f}"
              f"{errors.median():>20.1f}")

    # A false alert = an alert on a sensor that has no failure developing.
    false = test[test["alert"] & (test["label"] == "none")]
    starts = false[~((false["sensor_id"] == false["sensor_id"].shift(1)) &
                     (false["ts"] - false["ts"].shift(1) <= pd.Timedelta(minutes=STEP_MIN)))]
    days = (end - cut).total_seconds() / 86400
    print(f"\nfalse alerts in {days:.0f} test days: {len(starts)}")
    wrong_type = test[test["alert"] & test["label"].isin(TYPES) & (test["predicted"] != test["label"])]
    print(f"alerts that named the wrong failure type: {len(wrong_type)} of {int(test['alert'].sum())}")

    joblib.dump({"classifier": classifier, "regressors": regressors, "features": FEATURES,
                 "alert_probability": ALERT_PROBABILITY, "alert_in_a_row": ALERT_IN_A_ROW,
                 "step_min": STEP_MIN}, model_path)
    print(f"\nmodel saved to {model_path}")


if __name__ == "__main__":
    main(*sys.argv[1:])
'''

for path, text in FILES.items():
    full = os.path.join(ROOT, path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(text)
print("wrote", len(FILES), "files into", ROOT)

wrote 15 files into /kaggle/working/belt-monitor


In [10]:
!cd /kaggle/working/belt-monitor && rm -f data/train.db* && python -m simulator.run batch --days 60 --seed 42 --out data/train && python -m backend.load_csv data/train/readings.csv data/train.db && python -m model.train data/train.db data/train/failures.csv

readings written : 856001 of 864000 possible
seized_roller    : 27 episodes
belt_drift       : 29 episodes
belt_slip        : 30 episodes
outage           : 53
stuck            : 60
spike            : 1751
null_field       : 2516
dropped          : 4323
late             : 8478
{'received': 856001, 'stored': 856001, 'rejected': 0, 'duplicate': 0, 'late': 8478} in 73 s
history cut at 2026-10-13 02:56:30
train rows 120448, test rows 51000
  seized_roller  episodes: train 19, test 8
  belt_drift     episodes: train 21, test 8
  belt_slip      episodes: train 19, test 11

TEST RESULTS
failure type    episodes  warned model warning h   fixed limits h  hours-left error h
seized_roller          8       8            18.5             11.3                 0.4
belt_drift             8       8             6.0              2.5                 0.3
belt_slip             11      11             7.2              4.1                 0.1

false alerts in 18 test days: 0
alerts that named the wrong failure 

In [11]:
# Start the backend server in the background with an empty live database, then check it answers
import subprocess, time
!rm -f /kaggle/working/belt-monitor/data/belt.db*
server = subprocess.Popen(["python", "-m", "uvicorn", "backend.app:app", "--port", "8000"],
                          cwd="/kaggle/working/belt-monitor",
                          stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s http://127.0.0.1:8000/health

{"status":"ok","readings":0,"clock":null}

In [14]:
# Send 3 simulated days to the backend as a live feed, then list the failures the model predicted
!cd /kaggle/working/belt-monitor && python -m simulator.run live --seed 7 --days 3 --speedup 60000 --out data/live
!curl -s "http://127.0.0.1:8000/predictions?active=false" | python -c "import sys, json; [print(p['sensor_id'], p['failure_type'], 'predicted at', p['started_ts']) for p in reversed(json.load(sys.stdin))]"

messages sent: 42922, not delivered: 0
ALL belt_slip predicted at 2026-09-01T20:35:00Z
S3 seized_roller predicted at 2026-09-02T00:00:00Z
S8 seized_roller predicted at 2026-09-02T16:30:00Z


In [15]:
# Add the vision component (detector, loop logic, pretend belt) and the defects table and URLs in the backend
import os

ROOT = "/kaggle/working/belt-monitor"
FILES = {}

FILES["vision/__init__.py"] = r''''''

FILES["vision/tracker.py"] = r'''"""The loop logic: many detections over many laps become one record per damaged spot.

The belt is a closed loop, so a fixed camera sees every damaged spot again on
every lap. Without this file each lap would report the same damage as new.
"""
from collections import Counter

REPAIR = "patch work"            # an old repair: recorded, but never an alarm


def loop_distance(a, b, loop_m):
    """Shortest distance between two belt positions, going either way round the loop."""
    d = abs(a - b) % loop_m
    return min(d, loop_m - d)


def severity_of(kind, area_cm2):
    """An assumption, stated in the README: bigger is worse, and holes and tears are worse."""
    if kind == REPAIR:
        return "repair"
    step = 0 if area_cm2 < 50 else 1 if area_cm2 < 300 else 2
    if kind in ("Tear", "Hole"):
        step += 1
    return ["minor", "moderate", "severe", "severe"][step]


class DefectTracker:
    def __init__(self, loop_m, use_position=True, base_tolerance_m=1.0, drift_per_m=0.0,
                 lateral_tolerance_mm=250.0):
        self.loop_m = loop_m
        self.use_position = use_position            # False = the naive way: every detection is new
        self.base_tolerance_m = base_tolerance_m    # how far apart two sightings of one spot may be
        self.drift_per_m = drift_per_m              # extra tolerance per metre since the reference mark
        self.lateral_tolerance_mm = lateral_tolerance_mm
        self.records = []

    def add(self, lap, ts, position_m, lateral_mm, kind, area_cm2, confidence,
            since_mark_m=0.0, image=""):
        """Add one detection. Returns the record it now belongs to."""
        sighting = {"lap": lap, "ts": ts, "kind": kind, "area_cm2": area_cm2,
                    "confidence": confidence, "image": image}
        best = None
        if self.use_position:
            # The position estimate gets worse the further the belt has moved
            # since the reference mark, so the tolerance grows with that distance.
            tolerance = self.base_tolerance_m + self.drift_per_m * since_mark_m
            best_score = None
            for record in self.records:
                if record["sightings"][-1]["lap"] == lap:
                    continue                        # already seen this lap: it is a different spot
                distance = loop_distance(position_m, record["position_m"], self.loop_m)
                sideways = abs(lateral_mm - record["lateral_mm"])
                if distance > tolerance or sideways > self.lateral_tolerance_mm:
                    continue
                # Several records can be within reach. Take the closest one, counting
                # distance along the belt, distance across it, and a different type.
                score = distance / tolerance + sideways / self.lateral_tolerance_mm
                if record["sightings"][-1]["kind"] != kind:
                    score += 0.5
                if best is None or score < best_score:
                    best, best_score = record, score
        if best is None:
            best = {"id": f"D{len(self.records) + 1:03d}", "position_m": position_m % self.loop_m,
                    "lateral_mm": lateral_mm, "sightings": []}
            self.records.append(best)
        best["sightings"].append(sighting)
        return best

    def summary(self, current_lap):
        """One dictionary per damaged spot, ready for the backend and the dashboard."""
        out = []
        for record in self.records:
            seen = record["sightings"]
            kinds = Counter(s["kind"] for s in seen[-3:])            # the last 3 sightings vote
            kind = kinds.most_common(1)[0][0]
            first, last = seen[0], seen[-1]
            laps_seen = len({s["lap"] for s in seen})
            growth = None
            if laps_seen >= 3 and first["area_cm2"] > 0:
                growth = round(100.0 * (last["area_cm2"] - first["area_cm2"]) / first["area_cm2"], 1)
            if last["lap"] >= current_lap - 1:
                status = "new" if first["lap"] == current_lap and current_lap > 1 else "active"
            else:
                status = "not_seen"                 # missed for 2 laps or more: repaired, covered or a false alarm
            if kind == REPAIR and any(s["kind"] != REPAIR for s in seen[:-3]):
                status = "repaired"                 # it used to be damage, now it is a patch
            out.append({"id": record["id"], "position_m": round(record["position_m"], 1),
                        "lateral_mm": round(record["lateral_mm"]), "type": kind,
                        "severity": severity_of(kind, last["area_cm2"]),
                        "area_cm2": round(last["area_cm2"], 1), "growth_pct": growth,
                        "first_seen_ts": first["ts"], "last_seen_ts": last["ts"],
                        "laps_seen": laps_seen, "sightings": len(seen),
                        "confidence": round(last["confidence"], 2), "status": status,
                        "image": last["image"]})
        return out
'''

FILES["vision/detect.py"] = r'''"""Run the trained damage model on one image."""


def load_model(path):
    from ultralytics import YOLO
    return YOLO(path)


def detect(model, image, confidence=0.25):
    """Return (detections, result). Each detection says what was found and where in the frame.

    cx, cy and area_fraction are fractions of the frame (0 to 1), so they do
    not depend on the size of the photo.
    """
    result = model(image, conf=confidence, verbose=False)[0]
    detections = []
    if result.masks is None:
        return detections, result
    for outline, box in zip(result.masks.xyn, result.boxes):
        if len(outline) < 3:
            continue
        xs, ys = outline[:, 0], outline[:, 1]
        # Area inside the outline (shoelace formula).
        area = 0.5 * abs(float((xs * ys[list(range(1, len(ys))) + [0]]).sum()
                               - (ys * xs[list(range(1, len(xs))) + [0]]).sum()))
        detections.append({"kind": result.names[int(box.cls[0])],
                           "confidence": float(box.conf[0]),
                           "cx": float(xs.mean()), "cy": float(ys.mean()),
                           "area_fraction": area})
    return detections, result
'''

FILES["vision/virtual_belt.py"] = r'''"""A pretend belt built from the test photos, used to test the loop logic.

    python -m vision.virtual_belt /kaggle/working/belt_damage /kaggle/working/runs/belt_seg/weights/best.pt

Every test photo gets a fixed place on the 6 km loop. A pretend camera then
watches the belt go round several times. Because we placed the photos
ourselves, we know the true number of damaged spots, and can compare it with
the number of records the system creates.

No public dataset shows the same real damage over many laps, so the laps are
simulated: each lap the photo is made a little brighter or darker, and a few
frames are skipped as if material covered the belt.
"""
import glob
import json
import os
import random
import sys
import urllib.request
from datetime import datetime, timedelta

from simulator import config as C
from vision.tracker import DefectTracker

FRAME_LENGTH_M = 2.0         # assumption: one photo shows 2 m of belt, across the full width
SLOT_M = 3                   # photos are placed in 3 m slots, so two photos never overlap
LAPS = 5
COVERED_CHANCE = 0.05        # chance that a frame is hidden by material on a given lap
GROWING_FRAMES = 5           # this many photos are enlarged a little every lap (simulated growth)
GROWTH_PER_LAP = 0.06
START = datetime(2026, 9, 1)
OUT_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "data", "vision")


def build_belt(dataset_dir, seed=0):
    """Give every test photo a fixed place on the loop."""
    images = sorted(glob.glob(os.path.join(dataset_dir, "test", "images", "*")))
    rng = random.Random(seed)
    slots = rng.sample(range(C.BELT_LOOP_M // SLOT_M), len(images))
    frames = []
    for image, slot in zip(images, slots):
        label = os.path.join(dataset_dir, "test", "labels",
                             os.path.splitext(os.path.basename(image))[0] + ".txt")
        true_spots = sum(1 for line in open(label) if line.strip()) if os.path.exists(label) else 0
        frames.append({"image": image, "position_m": slot * SLOT_M + SLOT_M / 2,
                       "true_spots": true_spots, "growing": False})
    frames.sort(key=lambda f: f["position_m"])
    with_damage = [f for f in frames if f["true_spots"] > 0]
    for f in rng.sample(with_damage, min(GROWING_FRAMES, len(with_damage))):
        f["growing"] = True
    return frames


def lap_view(frame, lap, rng):
    """The photo as the camera sees it on this lap."""
    import cv2
    image = cv2.imread(frame["image"])
    image = cv2.convertScaleAbs(image, alpha=rng.uniform(0.75, 1.25))    # lighting changes
    if frame["growing"]:
        # Enlarge around the centre: the damage looks bigger each lap.
        zoom = 1 + GROWTH_PER_LAP * (lap - 1)
        h, w = image.shape[:2]
        big = cv2.resize(image, None, fx=zoom, fy=zoom)
        top, left = (big.shape[0] - h) // 2, (big.shape[1] - w) // 2
        image = big[top:top + h, left:left + w]
    return image


def watch(model, frames, detect, laps=LAPS, seed=0, save_last_lap=True):
    """Run the camera for some laps. Returns one observation per frame per lap."""
    rng = random.Random(seed)
    os.makedirs(os.path.join(OUT_DIR, "frames"), exist_ok=True)
    observations = []
    for lap in range(1, laps + 1):
        for number, frame in enumerate(frames):
            if rng.random() < COVERED_CHANCE:
                continue                                  # hidden under material this lap
            detections, result = detect(model, lap_view(frame, lap, rng))
            image_path = ""
            if save_last_lap and lap == laps and detections:
                image_path = os.path.join(OUT_DIR, "frames", f"frame_{number:03d}.jpg")
                result.save(filename=image_path)          # the photo with the outlines drawn on it
            observations.append({"lap": lap, "position_m": frame["position_m"],
                                 "detections": detections, "image": image_path})
    return observations


def track(observations, speed_error=0.0, seed=0, **tracker_settings):
    """Feed the observations to the loop logic.

    speed_error: each lap the belt position is estimated from speed x time, and the
    speed is off by about this share (0.001 = 0.1%). The error is zero at the
    reference mark (position 0, for example the belt splice) and grows from there.
    """
    rng = random.Random(seed)
    tracker = DefectTracker(C.BELT_LOOP_M, **tracker_settings)
    error_of_lap = {}
    for obs in observations:
        lap = obs["lap"]
        if lap not in error_of_lap:
            error_of_lap[lap] = rng.gauss(0, speed_error)
        estimate = obs["position_m"] * (1 + error_of_lap[lap])
        seconds = ((lap - 1) * C.BELT_LOOP_M + obs["position_m"]) / C.BELT_SPEED_MPS
        ts = (START + timedelta(seconds=seconds)).strftime("%Y-%m-%dT%H:%M:%SZ")
        for d in obs["detections"]:
            tracker.add(lap=lap, ts=ts,
                        position_m=estimate + (d["cy"] - 0.5) * FRAME_LENGTH_M,
                        lateral_mm=d["cx"] * C.BELT_WIDTH_MM, kind=d["kind"],
                        area_cm2=d["area_fraction"] * FRAME_LENGTH_M * 100 * C.BELT_WIDTH_MM / 10,
                        confidence=d["confidence"], since_mark_m=obs["position_m"],
                        image=obs["image"])
    return tracker


def report(frames, observations, laps=LAPS):
    true_spots = sum(f["true_spots"] for f in frames)
    per_lap = [sum(len(o["detections"]) for o in observations if o["lap"] == lap)
               for lap in range(1, laps + 1)]
    print(f"photos on the belt        : {len(frames)}")
    print(f"true damaged spots        : {true_spots}")
    print(f"detections on each lap    : {per_lap}")
    print(f"\nrecords created after {laps} laps (the true number is {true_spots})")
    print(f"{'position error per lap':24}{'rule off':>10}{'fixed 1 m':>11}{'growing':>9}")
    for error in [0.0, 0.001, 0.003]:
        off = track(observations, error, use_position=False)
        fixed = track(observations, error, base_tolerance_m=1.0)
        growing = track(observations, error, base_tolerance_m=1.0, drift_per_m=6 * error)
        print(f"{error * 100:>21.1f} %{len(off.records):>10}{len(fixed.records):>11}{len(growing.records):>9}")


def main(dataset_dir, weights, url="http://127.0.0.1:8000/defects"):
    from vision.detect import detect, load_model
    model = load_model(weights)
    frames = build_belt(dataset_dir)
    observations = watch(model, frames, detect)
    report(frames, observations)

    # The records shown on the dashboard: 0.1% position error, growing tolerance.
    tracker = track(observations, 0.001, base_tolerance_m=1.0, drift_per_m=0.006)
    records = tracker.summary(current_lap=LAPS)
    with open(os.path.join(OUT_DIR, "defects.json"), "w") as f:
        json.dump(records, f, indent=1)
    print(f"\n{len(records)} records saved to {os.path.join(OUT_DIR, 'defects.json')}")
    grown = sorted([r for r in records if r["growth_pct"] is not None],
                   key=lambda r: -r["growth_pct"])[:GROWING_FRAMES]
    for r in grown:
        print(f"  {r['id']} {r['type']:14} at {r['position_m']:7.1f} m  grew {r['growth_pct']:+.0f}% "
              f"over {r['laps_seen']} laps")
    try:
        request = urllib.request.Request(url, data=json.dumps(records).encode("utf-8"),
                                         headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(request, timeout=10) as response:
            print("sent to the backend:", response.read().decode())
    except OSError as error:
        print("backend not reachable, records were only saved to the file:", error)


if __name__ == "__main__":
    main(*sys.argv[1:])
'''

FILES["backend/db.py"] = r'''"""The database: one SQLite file with five tables."""
import os
import sqlite3

# The file sits in belt-monitor/data/belt.db unless BELT_DB says otherwise.
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DB_PATH = os.environ.get("BELT_DB") or os.path.join(PROJECT_DIR, "data", "belt.db")

SCHEMA = """
CREATE TABLE IF NOT EXISTS readings (
    sensor_id       TEXT NOT NULL,
    ts              TEXT NOT NULL,      -- when the sensor measured
    temperature_c   REAL,               -- NULL means missing or impossible
    vibration_mm_s  REAL,
    belt_speed_mps  REAL,
    drift_mm        REAL,
    flags           TEXT NOT NULL,      -- what was wrong with this reading, '' if nothing
    PRIMARY KEY (sensor_id, ts)         -- the same reading cannot be stored twice
);
CREATE INDEX IF NOT EXISTS readings_by_ts ON readings (ts);

CREATE TABLE IF NOT EXISTS rejected (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    reason  TEXT NOT NULL,
    raw     TEXT NOT NULL               -- the message exactly as it came in
);

CREATE TABLE IF NOT EXISTS anomalies (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id   TEXT NOT NULL,          -- 'ALL' for belt slip
    position_m  INTEGER NOT NULL,
    type        TEXT NOT NULL,
    severity    TEXT NOT NULL,          -- 'warning' or 'critical'
    value       REAL,                   -- the latest value that broke the limit
    started_ts  TEXT NOT NULL,
    last_ts     TEXT NOT NULL,
    ended_ts    TEXT                    -- NULL while the anomaly is still active
);

CREATE TABLE IF NOT EXISTS predictions (
    id                    INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id             TEXT NOT NULL,     -- 'ALL' for belt slip
    position_m            INTEGER NOT NULL,
    failure_type          TEXT NOT NULL,     -- what the model expects to fail
    probability           REAL NOT NULL,     -- how sure the model is, 0 to 1
    hours_left            REAL NOT NULL,     -- the model's estimate at last_ts
    predicted_failure_ts  TEXT NOT NULL,     -- last_ts + hours_left
    started_ts            TEXT NOT NULL,
    last_ts               TEXT NOT NULL,
    ended_ts              TEXT               -- NULL while the prediction is still active
);

CREATE TABLE IF NOT EXISTS defects (
    id             TEXT PRIMARY KEY,         -- one row per damaged spot on the belt
    position_m     REAL NOT NULL,            -- where on the 6 km loop
    lateral_mm     REAL NOT NULL,            -- how far across the belt
    type           TEXT NOT NULL,
    severity       TEXT NOT NULL,
    area_cm2       REAL NOT NULL,
    growth_pct     REAL,                     -- size change since first seen, NULL until 3 laps
    first_seen_ts  TEXT NOT NULL,
    last_seen_ts   TEXT NOT NULL,
    laps_seen      INTEGER NOT NULL,
    sightings      INTEGER NOT NULL,
    confidence     REAL NOT NULL,
    status         TEXT NOT NULL,            -- 'active', 'new', 'not_seen' or 'repaired'
    image          TEXT NOT NULL             -- path of the newest photo with the outline drawn
);
"""


def connect(path=None):
    path = path or DB_PATH
    os.makedirs(os.path.dirname(path), exist_ok=True)
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row          # rows can be read by column name
    conn.execute("PRAGMA journal_mode=WAL")  # the dashboard can read while the backend writes
    conn.executescript(SCHEMA)
    return conn
'''

FILES["backend/app.py"] = r'''"""The HTTP layer. Each function below answers one URL.

Start it from the belt-monitor folder with:
    python -m uvicorn backend.app:app --port 8000
"""
from fastapi import Body, FastAPI

from simulator import config as C
from backend.db import connect
from backend.ingest import ingest

app = FastAPI(title="Belt monitor")

PROCESS_ANOMALIES = ["high_temperature", "high_vibration", "belt_drift"]
SENSOR_FAULTS = ["sensor_offline", "sensor_stuck"]


@app.get("/health")
def health():
    conn = connect()
    count = conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
    clock = conn.execute("SELECT MAX(ts) FROM readings").fetchone()[0]
    conn.close()
    return {"status": "ok", "readings": count, "clock": clock}


@app.post("/readings")
def post_readings(messages: list[dict] = Body(...)):
    """The simulator sends its messages here. Each message is checked one by one,
    so one bad message cannot make the whole batch fail."""
    conn = connect()
    summary = ingest(conn, messages, with_predictions=True)
    conn.close()
    return summary


@app.get("/readings")
def get_readings(sensor_id: str, limit: int = 240):
    """The newest readings of one sensor, oldest first (for the dashboard charts)."""
    conn = connect()
    rows = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT ?",
                        (sensor_id, limit)).fetchall()
    conn.close()
    return [dict(r) for r in reversed(rows)]


@app.get("/anomalies")
def get_anomalies(active: bool = True, limit: int = 200):
    """Active anomalies, or the most recent ones of any state when active=false."""
    conn = connect()
    where = "WHERE ended_ts IS NULL" if active else ""
    rows = conn.execute(f"SELECT * FROM anomalies {where} ORDER BY started_ts DESC LIMIT ?",
                        (limit,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]


@app.get("/predictions")
def get_predictions(active: bool = True, limit: int = 200):
    """What the model expects to fail, where, and when."""
    conn = connect()
    where = "WHERE ended_ts IS NULL" if active else ""
    rows = conn.execute(f"SELECT * FROM predictions {where} ORDER BY started_ts DESC LIMIT ?",
                        (limit,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]


DEFECT_COLUMNS = ["id", "position_m", "lateral_mm", "type", "severity", "area_cm2", "growth_pct",
                  "first_seen_ts", "last_seen_ts", "laps_seen", "sightings", "confidence", "status", "image"]


@app.post("/defects")
def post_defects(records: list[dict] = Body(...)):
    """The vision component sends its current list of damaged spots. It replaces the old list."""
    conn = connect()
    conn.execute("DELETE FROM defects")
    for r in records:
        conn.execute("INSERT INTO defects VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                     [r.get(column) for column in DEFECT_COLUMNS])
    conn.commit()
    conn.close()
    return {"stored": len(records)}


@app.get("/defects")
def get_defects():
    """Wear and tear found by the vision component, one row per damaged spot."""
    conn = connect()
    rows = conn.execute("SELECT * FROM defects ORDER BY position_m").fetchall()
    conn.close()
    return [dict(r) for r in rows]


@app.get("/sensors")
def get_sensors():
    """One entry per sensor point: where it is, its newest reading and its status."""
    conn = connect()
    active = conn.execute("SELECT sensor_id, type, severity FROM anomalies "
                          "WHERE ended_ts IS NULL").fetchall()
    slip = [a for a in active if a["type"] == "belt_slip"]
    predicted = conn.execute("SELECT sensor_id, failure_type, hours_left FROM predictions "
                             "WHERE ended_ts IS NULL").fetchall()
    result = []
    for sid in C.SENSOR_IDS:
        newest = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT 1",
                              (sid,)).fetchone()
        mine = [a for a in active if a["sensor_id"] == sid]
        process = [a["type"] for a in mine if a["type"] in PROCESS_ANOMALIES]
        faults = [a["type"] for a in mine if a["type"] in SENSOR_FAULTS]
        if slip:
            process.append("belt_slip")        # slip slows the belt at every sensor point
        expected = [dict(p) for p in predicted if p["sensor_id"] in (sid, "ALL")]
        if newest is None or "sensor_offline" in faults:
            status = "no_data"                 # no current reading: it cannot be called normal
        elif process:
            status = "abnormal"                # a current reading is past its limit
        elif expected:
            status = "predicted_failure"       # inside the limits, but the model expects a failure
        else:
            status = "normal"
        result.append({"sensor_id": sid, "position_m": C.SENSOR_POSITION_M[sid],
                       "status": status, "anomalies": process, "sensor_faults": faults,
                       "predictions": expected, "newest": dict(newest) if newest else None})
    conn.close()
    return result
'''

for path, text in FILES.items():
    full = os.path.join(ROOT, path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(text)
print("wrote", len(FILES), "files into", ROOT)

wrote 6 files into /kaggle/working/belt-monitor


In [17]:
# Restart the backend server so it loads the new app.py with the /defects URLs
import subprocess, time
!pkill -f "[u]vicorn backend.app"
time.sleep(2)
server = subprocess.Popen(["python", "-m", "uvicorn", "backend.app:app", "--port", "8000"],
                          cwd="/kaggle/working/belt-monitor",
                          stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s http://127.0.0.1:8000/health

{"status":"ok","readings":42922,"clock":"2026-09-03T23:59:00Z"}

In [18]:
!curl -s http://127.0.0.1:8000/defects

[]

In [19]:
!cd /kaggle/working/belt-monitor && python -m vision.virtual_belt /kaggle/working/belt_damage /kaggle/working/runs/belt_seg/weights/best.pt

photos on the belt        : 92
true damaged spots        : 143
detections on each lap    : [133, 131, 133, 137, 138]

records created after 5 laps (the true number is 143)
position error per lap    rule off  fixed 1 m  growing
                  0.0 %       672        148      148
                  0.1 %       672        555      148
                  0.3 %       672        621      147

148 records saved to /kaggle/working/belt-monitor/data/vision/defects.json
  D122 impact damage  at  5547.6 m  grew +434% over 4 laps
  D083 Tear           at  4337.9 m  grew +93% over 5 laps
  D085 Tear           at  4337.6 m  grew +89% over 5 laps
  D086 Tear           at  4337.4 m  grew +75% over 5 laps
  D084 Tear           at  4338.2 m  grew +69% over 5 laps
sent to the backend: {"stored":148}


In [20]:
import os

ROOT = "/kaggle/working/belt-monitor"
FILES = {}

FILES["dashboard/app.py"] = r'''"""Belt health dashboard.

Start it from the belt-monitor folder with:
    python -m streamlit run dashboard/app.py --server.port 8501

It only reads from the backend's HTTP API, the same way any other program could.
"""
import json
import os
import urllib.request

import pandas as pd
import streamlit as st

API = os.environ.get("BELT_API", "http://127.0.0.1:8000")
DOT = {"normal": "🟢", "abnormal": "🔴", "predicted_failure": "🟠", "no_data": "⚪"}
WORD = {"normal": "normal", "abnormal": "abnormal", "predicted_failure": "predicted failure",
        "no_data": "no data"}


def get(path):
    with urllib.request.urlopen(API + path, timeout=10) as response:
        return json.load(response)


def where(sensor_id, position_m):
    if sensor_id == "ALL":
        return f"drive pulley, {position_m} m"
    return f"{sensor_id}, {position_m} m"


st.set_page_config(page_title="Belt monitor", layout="wide")
st.title("Conveyor belt health")

try:
    health = get("/health")
except OSError as error:
    st.error(f"The backend is not reachable at {API}: {error}")
    st.stop()

sensors = get("/sensors")
anomalies = get("/anomalies")
predictions = get("/predictions")
defects = get("/defects")

st.caption(f"Belt time: {health['clock']}  |  3 km conveyor, 6 km belt loop, 4 m/s  |  "
           f"{health['readings']} readings stored")
if st.button("Refresh"):
    st.rerun()

faults = [a for a in anomalies if a["type"] in ("sensor_offline", "sensor_stuck")]
process = [a for a in anomalies if a not in faults]
open_damage = [d for d in defects if d["severity"] != "repair" and d["status"] in ("active", "new")]
top = st.columns(5)
top[0].metric("Sensor points normal", sum(1 for s in sensors if s["status"] == "normal"))
top[1].metric("Active anomalies", len(process))
top[2].metric("Predicted failures", len(predictions))
top[3].metric("Sensor faults", len(faults))
top[4].metric("Damaged spots on the belt", len(open_damage))

# ---------------------------------------------------------------- 1. sensor points
st.header("1. Sensor points")
st.caption("🟢 normal   🔴 abnormal: a reading is past its limit   "
           "🟠 predicted failure: inside the limits, but the model expects a failure   "
           "⚪ no data: the sensor is silent")
for column, s in zip(st.columns(len(sensors)), sensors):
    column.markdown(f"### {DOT[s['status']]} {s['sensor_id']}")
    column.caption(f"{s['position_m']} m")
    column.write(WORD[s["status"]])
    reading = s["newest"] or {}
    lines = []
    for label, key, unit in [("T", "temperature_c", "°C"), ("V", "vibration_mm_s", "mm/s"),
                             ("S", "belt_speed_mps", "m/s"), ("D", "drift_mm", "mm")]:
        value = reading.get(key)
        lines.append(f"{label}: {'missing' if value is None else f'{value:.1f} {unit}'}")
    column.caption("  \n".join(lines))
    if s["sensor_faults"]:
        column.caption("fault: " + ", ".join(f.replace("sensor_", "") for f in s["sensor_faults"]))

# ---------------------------------------------------------------- 2. anomalies
st.header("2. Active anomalies")
if process:
    st.dataframe(pd.DataFrame([{"location": where(a["sensor_id"], a["position_m"]),
                                "type": a["type"], "severity": a["severity"],
                                "latest value": a["value"], "since": a["started_ts"]}
                               for a in process]), hide_index=True)
else:
    st.success("No reading is past its limit.")
if faults:
    st.subheader("Sensor faults (the data cannot be trusted)")
    st.dataframe(pd.DataFrame([{"location": where(a["sensor_id"], a["position_m"]),
                                "type": a["type"], "since": a["started_ts"]}
                               for a in faults]), hide_index=True)

# ---------------------------------------------------------------- 3. predictions
st.header("3. Model predictions")
if predictions:
    st.dataframe(pd.DataFrame([{"what might fail": p["failure_type"],
                                "where": where(p["sensor_id"], p["position_m"]),
                                "when (estimate)": p["predicted_failure_ts"],
                                "hours left": round(p["hours_left"], 1),
                                "how sure": f"{p['probability'] * 100:.0f} %",
                                "predicted since": p["started_ts"]}
                               for p in predictions]), hide_index=True)
else:
    st.success("The model expects no failure.")

# ---------------------------------------------------------------- 4. vision
st.header("4. Vision: wear and tear on the belt")
st.caption("One row per damaged spot, however many laps the camera has seen it. "
           "Position is measured along the 6 km loop from the reference mark.")
if defects:
    table = pd.DataFrame(defects)
    by_type = table[table["severity"] != "repair"]["type"].value_counts()
    st.write("Damaged spots by type: " + ", ".join(f"{name} {count}" for name, count in by_type.items())
             + f"  |  old repairs (patch work): {int((table['severity'] == 'repair').sum())}")
    order = {"severe": 0, "moderate": 1, "minor": 2, "repair": 3}
    table["rank"] = table["severity"].map(order)
    table = table.sort_values(["rank", "growth_pct"], ascending=[True, False])
    st.dataframe(table[["id", "position_m", "lateral_mm", "type", "severity", "area_cm2",
                        "growth_pct", "laps_seen", "status", "last_seen_ts"]], hide_index=True)
    with_photo = [d for d in table.to_dict("records") if d["image"] and os.path.exists(d["image"])]
    if with_photo:
        names = [f"{d['id']}  {d['type']}  at {d['position_m']} m" for d in with_photo]
        chosen = st.selectbox("Show the newest photo of", names)
        st.image(with_photo[names.index(chosen)]["image"], width=520)
else:
    st.info("The vision component has not sent any records yet.")

# ---------------------------------------------------------------- 5. history
st.header("5. One sensor over the last 4 hours")
sensor_id = st.selectbox("Sensor", [s["sensor_id"] for s in sensors])
history = pd.DataFrame(get(f"/readings?sensor_id={sensor_id}&limit=240"))
if len(history):
    history["ts"] = pd.to_datetime(history["ts"])
    history = history.set_index("ts")
    left, right = st.columns(2)
    left.caption("Temperature (°C). Warning above 70, critical above 85.")
    left.line_chart(history["temperature_c"])
    right.caption("Vibration (mm/s). Warning above 4.5, critical above 7.1.")
    right.line_chart(history["vibration_mm_s"])
    left.caption("Belt speed (m/s). Warning below 3.8, critical below 3.4.")
    left.line_chart(history["belt_speed_mps"])
    right.caption("Sideways drift (mm). Warning beyond 75, critical beyond 120.")
    right.line_chart(history["drift_mm"])

with st.expander("Recently closed anomalies and predictions"):
    past = [a for a in get("/anomalies?active=false&limit=30") if a["ended_ts"]]
    st.dataframe(pd.DataFrame([{"location": where(a["sensor_id"], a["position_m"]), "type": a["type"],
                                "severity": a["severity"], "from": a["started_ts"], "to": a["ended_ts"]}
                               for a in past]), hide_index=True)
    past = [p for p in get("/predictions?active=false&limit=30") if p["ended_ts"]]
    st.dataframe(pd.DataFrame([{"what": p["failure_type"], "where": where(p["sensor_id"], p["position_m"]),
                                "from": p["started_ts"], "to": p["ended_ts"]} for p in past]),
                 hide_index=True)
'''

FILES["backend/app.py"] = r'''"""The HTTP layer. Each function below answers one URL.

Start it from the belt-monitor folder with:
    python -m uvicorn backend.app:app --port 8000
"""
from fastapi import Body, FastAPI

from simulator import config as C
from backend.db import connect
from backend.ingest import ingest

app = FastAPI(title="Belt monitor")

PROCESS_ANOMALIES = ["high_temperature", "high_vibration", "belt_drift", "belt_slip"]
SENSOR_FAULTS = ["sensor_offline", "sensor_stuck"]
DRIVE_SENSOR = "S10"         # belt slip happens at the drive pulley, which is where S10 sits


@app.get("/health")
def health():
    conn = connect()
    count = conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
    clock = conn.execute("SELECT MAX(ts) FROM readings").fetchone()[0]
    conn.close()
    return {"status": "ok", "readings": count, "clock": clock}


@app.post("/readings")
def post_readings(messages: list[dict] = Body(...)):
    """The simulator sends its messages here. Each message is checked one by one,
    so one bad message cannot make the whole batch fail."""
    conn = connect()
    summary = ingest(conn, messages, with_predictions=True)
    conn.close()
    return summary


@app.get("/readings")
def get_readings(sensor_id: str, limit: int = 240):
    """The newest readings of one sensor, oldest first (for the dashboard charts)."""
    conn = connect()
    rows = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT ?",
                        (sensor_id, limit)).fetchall()
    conn.close()
    return [dict(r) for r in reversed(rows)]


@app.get("/anomalies")
def get_anomalies(active: bool = True, limit: int = 200):
    """Active anomalies, or the most recent ones of any state when active=false."""
    conn = connect()
    where = "WHERE ended_ts IS NULL" if active else ""
    rows = conn.execute(f"SELECT * FROM anomalies {where} ORDER BY started_ts DESC LIMIT ?",
                        (limit,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]


@app.get("/predictions")
def get_predictions(active: bool = True, limit: int = 200):
    """What the model expects to fail, where, and when."""
    conn = connect()
    where = "WHERE ended_ts IS NULL" if active else ""
    rows = conn.execute(f"SELECT * FROM predictions {where} ORDER BY started_ts DESC LIMIT ?",
                        (limit,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]


DEFECT_COLUMNS = ["id", "position_m", "lateral_mm", "type", "severity", "area_cm2", "growth_pct",
                  "first_seen_ts", "last_seen_ts", "laps_seen", "sightings", "confidence", "status", "image"]


@app.post("/defects")
def post_defects(records: list[dict] = Body(...)):
    """The vision component sends its current list of damaged spots. It replaces the old list."""
    conn = connect()
    conn.execute("DELETE FROM defects")
    for r in records:
        conn.execute("INSERT INTO defects VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                     [r.get(column) for column in DEFECT_COLUMNS])
    conn.commit()
    conn.close()
    return {"stored": len(records)}


@app.get("/defects")
def get_defects():
    """Wear and tear found by the vision component, one row per damaged spot."""
    conn = connect()
    rows = conn.execute("SELECT * FROM defects ORDER BY position_m").fetchall()
    conn.close()
    return [dict(r) for r in rows]


@app.get("/sensors")
def get_sensors():
    """One entry per sensor point: where it is, its newest reading and its status."""
    conn = connect()
    active = conn.execute("SELECT sensor_id, type, severity FROM anomalies "
                          "WHERE ended_ts IS NULL").fetchall()
    predicted = conn.execute("SELECT sensor_id, failure_type, hours_left FROM predictions "
                             "WHERE ended_ts IS NULL").fetchall()
    result = []
    for sid in C.SENSOR_IDS:
        newest = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT 1",
                              (sid,)).fetchone()
        # Belt slip is stored with sensor 'ALL'. It is shown at the drive pulley sensor.
        names = (sid, "ALL") if sid == DRIVE_SENSOR else (sid,)
        mine = [a for a in active if a["sensor_id"] in names]
        process = [a["type"] for a in mine if a["type"] in PROCESS_ANOMALIES]
        faults = [a["type"] for a in mine if a["type"] in SENSOR_FAULTS]
        expected = [dict(p) for p in predicted if p["sensor_id"] in names]
        if newest is None or "sensor_offline" in faults:
            status = "no_data"                 # no current reading: it cannot be called normal
        elif process:
            status = "abnormal"                # a current reading is past its limit
        elif expected:
            status = "predicted_failure"       # inside the limits, but the model expects a failure
        else:
            status = "normal"
        result.append({"sensor_id": sid, "position_m": C.SENSOR_POSITION_M[sid],
                       "status": status, "anomalies": process, "sensor_faults": faults,
                       "predictions": expected, "newest": dict(newest) if newest else None})
    conn.close()
    return result
'''

FILES["simulator/run.py"] = r'''"""Command line for the simulator.

History for training (writes three CSV files):
    python -m simulator.run batch --days 60 --seed 42 --out data/train

Live feed for the backend (one HTTP POST per simulated minute):
    python -m simulator.run live --url http://127.0.0.1:8000/readings --seed 7 --speedup 600
"""
import argparse
import csv
import json
import os
import time
import urllib.error
import urllib.request

from simulator import config as C
from simulator.generate import BeltSimulator

FAILURE_COLUMNS = ["failure_id", "type", "sensor_id", "onset_ts", "failure_ts", "repaired_ts"]
FAULT_COLUMNS = ["fault_type", "sensor_id", "channel", "start_ts", "end_ts", "detail"]


def write_csv(path, columns, rows):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def run_batch(args):
    sim = BeltSimulator(args.days, args.seed, args.start, messy=not args.clean)
    os.makedirs(args.out, exist_ok=True)
    count = 0
    with open(os.path.join(args.out, "readings.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["arrival_ts", "sensor_id", "ts"] + C.CHANNELS)
        for arrival_ts, messages in sim.ticks():
            for msg in messages:
                # A null value is written as an empty cell.
                row = [arrival_ts, msg["sensor_id"], msg["ts"]]
                for ch in C.CHANNELS:
                    row.append("" if msg[ch] is None else msg[ch])
                writer.writerow(row)
                count += 1
    write_csv(os.path.join(args.out, "failures.csv"), FAILURE_COLUMNS, sim.failures)
    write_csv(os.path.join(args.out, "data_faults.csv"), FAULT_COLUMNS, sim.data_faults)

    expected = sim.total_min * len(C.SENSOR_IDS)
    print(f"readings written : {count} of {expected} possible")
    for ftype in C.FAILURE_DURATION_H:
        print(f"{ftype:17}: {sum(1 for x in sim.failures if x['type'] == ftype)} episodes")
    for kind in ["outage", "stuck", "spike", "null_field", "dropped", "late"]:
        print(f"{kind:17}: {sum(1 for x in sim.data_faults if x['fault_type'] == kind)}")


def post_json(url, payload):
    body = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=body,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=5) as response:
        return response.status


def run_live(args):
    sim = BeltSimulator(args.days, args.seed, args.start, messy=not args.clean)
    os.makedirs(args.out, exist_ok=True)
    # The failure schedule is known before the first reading, so write it now.
    write_csv(os.path.join(args.out, "failures.csv"), FAILURE_COLUMNS, sim.failures)
    pause = C.READING_INTERVAL_S / args.speedup      # real seconds per simulated minute
    sent = failed = 0
    try:
        for minute, (arrival_ts, messages) in enumerate(sim.ticks()):
            if args.stop_after is not None and minute >= args.stop_after * 24 * 60:
                print(f"stopped on purpose at {arrival_ts}")
                break
            if messages:
                try:
                    post_json(args.url, messages)
                    sent += len(messages)
                except (urllib.error.URLError, OSError) as error:
                    failed += len(messages)
                    print(f"{arrival_ts} POST failed: {error}")
            time.sleep(pause)
    except KeyboardInterrupt:
        print("stopped by user")
    finally:
        write_csv(os.path.join(args.out, "data_faults.csv"), FAULT_COLUMNS, sim.data_faults)
        print(f"messages sent: {sent}, not delivered: {failed}")


def main():
    parser = argparse.ArgumentParser(description="Conveyor belt sensor simulator")
    sub = parser.add_subparsers(dest="mode", required=True)
    for name in ["batch", "live"]:
        p = sub.add_parser(name)
        p.add_argument("--days", type=float, default=60)
        p.add_argument("--seed", type=int, default=42)
        p.add_argument("--start", default="2026-09-01T00:00:00Z")
        p.add_argument("--clean", action="store_true", help="no messy data, failures only")
        p.add_argument("--out", default="data/train" if name == "batch" else "data/live")
        if name == "live":
            p.add_argument("--url", default="http://127.0.0.1:8000/readings")
            p.add_argument("--speedup", type=float, default=600,
                           help="600 means one simulated minute takes 0.1 real seconds")
            p.add_argument("--stop-after", type=float, default=None,
                           help="stop sending after this many simulated days (the run itself "
                                "is still planned for --days, so the data stays the same)")
    args = parser.parse_args()
    if args.mode == "batch":
        run_batch(args)
    else:
        run_live(args)


if __name__ == "__main__":
    main()
'''

for path, text in FILES.items():
    full = os.path.join(ROOT, path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(text)
print("wrote", len(FILES), "files into", ROOT)

wrote 3 files into /kaggle/working/belt-monitor


In [21]:
# Restart the server with an empty database and replay the live feed up to day 1.2, the moment shown in the dashboard screenshots
import subprocess, time
!pkill -f "[u]vicorn backend.app"
time.sleep(2)
!rm -f /kaggle/working/belt-monitor/data/belt.db*
server = subprocess.Popen(["python", "-m", "uvicorn", "backend.app:app", "--port", "8000"],
                          cwd="/kaggle/working/belt-monitor",
                          stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!cd /kaggle/working/belt-monitor && python -m simulator.run live --seed 7 --days 3 --stop-after 1.2 --speedup 60000 --out data/live
!cd /kaggle/working/belt-monitor && curl -s -X POST -H "Content-Type: application/json" --data @data/vision/defects.json http://127.0.0.1:8000/defects
!curl -s http://127.0.0.1:8000/health

stopped on purpose at 2026-09-02T04:48:00Z
messages sent: 17121, not delivered: 0
{"stored":148}{"status":"ok","readings":17121,"clock":"2026-09-02T04:47:00Z"}

In [22]:
# Start the dashboard and open a temporary public address for it (Since,Kaggle has no direct way to view a web page)
!pip install -q streamlit
!wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /kaggle/working/cloudflared
import subprocess, time, re
dash = subprocess.Popen(["python", "-m", "streamlit", "run", "dashboard/app.py", "--server.port", "8501",
                         "--server.headless", "true", "--server.enableCORS", "false",
                         "--server.enableXsrfProtection", "false", "--browser.gatherUsageStats", "false"],
                        cwd="/kaggle/working/belt-monitor",
                        stdout=open("/kaggle/working/dashboard.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--protocol", "http2",
                           "--url", "http://localhost:8501"],
                          stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT)
time.sleep(15)
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read())
print(urls[0] if urls else "no address yet")

https://conferences-trades-retained-syntax.trycloudflare.com


In [23]:
# Two improvements after looking at the dashboard: charts with limit lines, and growth measured on same-type sightings only
ROOT = "/kaggle/working/belt-monitor"

NEW_CHARTS = r'''# ---------------------------------------------------------------- 5. history
PANELS = [("temperature_c", "Temperature (°C)", [70, 85]),
          ("vibration_mm_s", "Vibration (mm/s)", [4.5, 7.1]),
          ("belt_speed_mps", "Belt speed (m/s)", [3.8, 3.4]),
          ("drift_mm", "Sideways drift (mm)", [75, 120, -75, -120])]


def history_figure(history):
    """Four small charts. Dashed lines are the limits: orange = warning, red = critical."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 6.5), sharex=True)
    for ax, (column, title, limits) in zip(axes.flat, PANELS):
        ax.plot(history.index, history[column], linewidth=1.2, color="#1f77b4")
        for number, limit in enumerate(limits):
            ax.axhline(limit, linestyle="--", linewidth=1, color="orange" if number % 2 == 0 else "red")
        ax.set_title(title, loc="left")
        ax.grid(alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    fig.tight_layout()
    return fig


st.header("5. One sensor over the last 4 hours")
# Sensors that are not normal come first in the list.
names = [s["sensor_id"] for s in sorted(sensors, key=lambda s: s["status"] == "normal")]
sensor_id = st.selectbox("Sensor", names)
history = pd.DataFrame(get(f"/readings?sensor_id={sensor_id}&limit=240"))
if len(history):
    # Belt time, without converting to the viewer's time zone.
    history["ts"] = pd.to_datetime(history["ts"]).dt.tz_localize(None)
    history = history.set_index("ts")
    st.caption("Dashed lines are the limits: orange = warning, red = critical. Times are belt time.")
    st.pyplot(history_figure(history))

'''

OLD_GROWTH = r'''            growth = None
            if laps_seen >= 3 and first["area_cm2"] > 0:
                growth = round(100.0 * (last["area_cm2"] - first["area_cm2"]) / first["area_cm2"], 1)
'''

NEW_GROWTH = r'''            # Growth compares like with like: only sightings of the same type count.
            # Otherwise a spot seen once as a small tear and later as a large area of
            # impact damage would look as if it had grown.
            same_type = [s for s in seen if s["kind"] == kind]
            growth = None
            if len({s["lap"] for s in same_type}) >= 3 and same_type[0]["area_cm2"] > 0:
                growth = round(100.0 * (same_type[-1]["area_cm2"] - same_type[0]["area_cm2"])
                               / same_type[0]["area_cm2"], 1)
'''

# 1. Dashboard: fixed charts with the limits drawn in.
path = ROOT + "/dashboard/app.py"
text = open(path).read()
start = text.index("# ---------------------------------------------------------------- 5. history")
end = text.index('with st.expander("Recently closed anomalies and predictions"):')
text = text[:start] + NEW_CHARTS + text[end:]
if "import matplotlib" not in text:
    text = text.replace("import pandas as pd\n", 'import matplotlib\nmatplotlib.use("Agg")\n'
                        "import matplotlib.dates as mdates\nimport matplotlib.pyplot as plt\nimport pandas as pd\n", 1)
open(path, "w").write(text)
print("dashboard updated")

# 2. Vision: growth compares sightings of the same type only.
path = ROOT + "/vision/tracker.py"
text = open(path).read()
if OLD_GROWTH in text:
    open(path, "w").write(text.replace(OLD_GROWTH, NEW_GROWTH))
    print("tracker updated")
else:
    print("tracker was already updated")

!cd /kaggle/working/belt-monitor && python -m vision.virtual_belt /kaggle/working/belt_damage /kaggle/working/runs/belt_seg/weights/best.pt

dashboard updated
tracker updated
photos on the belt        : 92
true damaged spots        : 143
detections on each lap    : [133, 131, 133, 137, 138]

records created after 5 laps (the true number is 143)
position error per lap    rule off  fixed 1 m  growing
                  0.0 %       672        148      148
                  0.1 %       672        555      148
                  0.3 %       672        621      147

148 records saved to /kaggle/working/belt-monitor/data/vision/defects.json
  D122 impact damage  at  5547.6 m  grew +434% over 4 laps
  D083 Tear           at  4337.9 m  grew +93% over 5 laps
  D085 Tear           at  4337.6 m  grew +89% over 5 laps
  D086 Tear           at  4337.4 m  grew +75% over 5 laps
  D084 Tear           at  4338.2 m  grew +69% over 5 laps
sent to the backend: {"stored":148}


In [24]:
!cd /kaggle/working/belt-monitor && python -m simulator.run live --seed 7 --days 3 --stop-after 1.75 --speedup 60000 --out data/live

stopped on purpose at 2026-09-02T18:00:00Z
messages sent: 25005, not delivered: 0
